> title : 제 4회 ETRI 휴먼이해 인공지능 논문경진대회 <br>
> author : 단머스 8기 <br>

### 📦 라이브러리

In [1]:
import os
import random
import re
import warnings
from pathlib import Path
from enum import Enum
from collections import Counter
from functools import reduce

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from category_encoders import TargetEncoder

from lightgbm import LGBMClassifier, early_stopping


### ⚙️ 시스템 세팅

In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(1)

In [3]:
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 999)
pd.set_option('display.max_rows', 999)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: '%0.4f' % x)

### ⚙️ 전역 파라미터 설정

In [4]:
# DATA_DIR = Path("/kaggle/input/dacon-etri-lifelog/ETRI_lifelog_dataset")
DATA_DIR = Path("./data/")

In [5]:
SLEEP_HOURS = tuple(range(0, 5)) ### 수정
MIGHT_GO_TO_SLEEP_HOURS = tuple(range(20, 24)) + tuple(range(0, 2))
MIGHT_WAKEUP_HOURS = tuple(range(6, 10))
ACTIVE_HOURS = tuple(range(7, 24))
WORK_HOURS = tuple(range(7, 19))
FREE_HOURS = tuple(range(19, 24))

HOLIDAY_DATES = [
    pd.Timestamp('2024-08-15'),
    pd.Timestamp('2024-09-16'),
    pd.Timestamp('2024-09-17'),
    pd.Timestamp('2024-09-18'),
    pd.Timestamp('2024-10-03'),
    pd.Timestamp('2024-10-09'),
]

### ⚒️ 데이터 유틸리티

In [6]:
class DataType(Enum):
    mACStatus = "mACStatus"
    mActivity = "mActivity"
    mAmbience = "mAmbience"
    mBle = "mBle"
    mGps = "mGps"
    mLight = "mLight"
    mScreenStatus = "mScreenStatus"
    mUsageStats = "mUsageStats"
    mWifi = "mWifi"
    wHr = "wHr"
    wLight = "wLight"
    wPedo = "wPedo"

In [7]:
def load_data(data_type: DataType):
    file_path = DATA_DIR / f"ch2025_data_items/ch2025_{data_type.value}.parquet"
    df = pd.read_parquet(file_path)
    df["subject_id"] = df["subject_id"].astype("category")
    df["lifelog_date"] = df["timestamp"].dt.normalize()
    df["month"] = df["timestamp"].dt.month
    df["day"] = df["timestamp"].dt.day
    df["hour"] = df["timestamp"].dt.hour
    df["minute"] = df["timestamp"].dt.minute
    df["weekday"] = df["timestamp"].dt.weekday
    fixed_columns = ["subject_id", "timestamp", "lifelog_date", "month", "day", "hour", "minute", "weekday"]
    columns = df.columns.tolist()
    columns = fixed_columns + [col for col in columns if col not in fixed_columns]
    df = df[columns]
    df = df.sort_values(by=["subject_id", "timestamp"])
    return df

def load_train():
    df = pd.read_csv(DATA_DIR / "ch2025_metrics_train.csv")
    df["subject_id"] = df["subject_id"].astype("category")
    df["sleep_date"] = pd.to_datetime(df["sleep_date"]).dt.normalize()
    df["lifelog_date"] = pd.to_datetime(df["lifelog_date"]).dt.normalize()
    return df


def load_val():
    from io import StringIO
    train_df = load_train()
    val_ids = "subject_id,sleep_date\nid01,2024-07-24\nid01,2024-07-27\nid01,2024-08-18\nid01,2024-08-19\nid01,2024-08-20\nid01,2024-08-21\nid01,2024-08-22\nid01,2024-08-24\nid01,2024-08-25\nid01,2024-08-26\nid01,2024-08-27\nid01,2024-08-28\nid01,2024-08-29\nid01,2024-08-30\nid02,2024-08-23\nid02,2024-08-24\nid02,2024-09-16\nid02,2024-09-17\nid02,2024-09-19\nid02,2024-09-20\nid02,2024-09-21\nid02,2024-09-22\nid02,2024-09-23\nid02,2024-09-24\nid02,2024-09-25\nid02,2024-09-26\nid02,2024-09-27\nid02,2024-09-28\nid03,2024-08-30\nid03,2024-09-01\nid03,2024-09-02\nid03,2024-09-03\nid03,2024-09-05\nid03,2024-09-06\nid03,2024-09-07\nid04,2024-09-03\nid04,2024-09-04\nid04,2024-09-05\nid04,2024-09-06\nid04,2024-09-07\nid04,2024-09-08\nid04,2024-09-09\nid04,2024-10-08\nid04,2024-10-09\nid04,2024-10-10\nid04,2024-10-11\nid04,2024-10-12\nid04,2024-10-13\nid04,2024-10-14\nid05,2024-10-19\nid05,2024-10-23\nid05,2024-10-24\nid05,2024-10-25\nid05,2024-10-26\nid05,2024-10-27\nid05,2024-10-28\nid06,2024-07-25\nid06,2024-07-26\nid06,2024-07-27\nid06,2024-07-28\nid06,2024-07-29\nid06,2024-07-30\nid06,2024-07-31\nid07,2024-07-07\nid07,2024-07-08\nid07,2024-07-09\nid07,2024-07-10\nid07,2024-07-11\nid07,2024-07-12\nid07,2024-07-13\nid07,2024-07-30\nid07,2024-08-01\nid07,2024-08-02\nid07,2024-08-03\nid07,2024-08-04\nid07,2024-08-05\nid07,2024-08-06\nid08,2024-08-28\nid08,2024-08-29\nid08,2024-08-30\nid08,2024-08-31\nid08,2024-09-01\nid08,2024-09-02\nid08,2024-09-04\nid09,2024-08-02\nid09,2024-08-22\nid09,2024-08-23\nid09,2024-08-24\nid09,2024-08-25\nid09,2024-08-27\nid09,2024-08-28\nid09,2024-08-29\nid09,2024-08-30\nid09,2024-08-31\nid09,2024-09-01\nid09,2024-09-02\nid09,2024-09-03\nid09,2024-09-04\nid10,2024-08-28\nid10,2024-08-30\nid10,2024-08-31\nid10,2024-09-01\nid10,2024-09-02\nid10,2024-09-03\nid10,2024-09-06\n"
    val_df = pd.read_csv(StringIO(val_ids))
    val_df = val_df.astype({"subject_id": "category", "sleep_date": "datetime64[ns]"})
    val_df = train_df.merge(val_df, on=["subject_id", "sleep_date"], how="inner")
    return val_df


def load_test():
    df = pd.read_csv(DATA_DIR / "ch2025_submission_sample.csv")
    df["subject_id"] = df["subject_id"].astype("category")
    df["sleep_date"] = pd.to_datetime(df["sleep_date"]).dt.normalize()
    df["lifelog_date"] = pd.to_datetime(df["lifelog_date"]).dt.normalize()
    return df

In [8]:
def describe_df(df):
    print(f"# shape:\n{df.shape}\n")
    print(f"# dtypes:\n{df.dtypes}\n")
    # print(f"# head:\n{df.head(3)}\n")
    display(df.head(3))
    nan_stats = df.isna().sum().to_frame(name='missing_count')
    nan_stats['missing_ratio(%)'] = (df.isna().mean() * 100).round(2)
    print(f"# nan_stats:\n" + nan_stats.to_string() + "\n")

In [9]:
def shift_lifelog_date(df, target_hours):
    df = df.copy()
    mask = df["hour"].isin(target_hours) & df["hour"].lt(12)
    df.loc[mask, "lifelog_date"] = df.loc[mask, "lifelog_date"] - pd.Timedelta(days=1)
    df.loc[mask, "day"] = df.loc[mask, "day"] - 1
    df = df.sort_values(by=["subject_id", "lifelog_date", "timestamp"])
    return df

In [10]:
def datetime_to_minute(dt):
    hour = dt.hour
    minute = dt.minute

    if hour < 12:
        return (hour + 24) * 60 + minute
    if hour >= 12:
        return hour * 60 + minute

### ✔️ mACStatus 핸드폰 충전상태
- Indicates whether the smartphone is currently being charged.
- m_charging : 0/1 상태
- 핸드폰이 오랫 동안 충전했다는 의미?
 - 한 자리에 장시간 머물러 있었다.
 - 핸드폰을 장시간 사용하지 않았다.

In [11]:
def run_length_encoding(arr):
    """Run-Length Encoding"""
    if len(arr) == 0:
        return []

    diffs = np.diff(np.concatenate(([0], arr, [0])))
    run_starts = np.where(diffs == 1)[0]
    run_ends = np.where(diffs == -1)[0]
    return run_ends - run_starts

def process_mACStatus(df):
    status = df["m_charging"].values

    def _process_feature(status):
        if len(status) == 0:
            return 0., 0., 0., 0., 0.

        # charging 상태 비율, 합
        ratio_charging = status.mean()
        sum_charging = status.sum()

        # 상태전이 횟수
        transitions = (status[1:] != status[:-1]).sum()

        lengths = run_length_encoding(status)
        avg_charging_duration = np.mean(lengths) if len(lengths) > 0 else 0
        max_charging_duration = np.max(lengths) if len(lengths) > 0 else 0

        return ratio_charging, sum_charging, transitions, avg_charging_duration, max_charging_duration

    # 하루
    charging_ratio, charging_sum, chargning_transitions, avg_charging_duration, max_charging_duration = _process_feature(status)

    # 잠자는 시간대
    sleep_status = status[df["hour"].isin(SLEEP_HOURS)]
    sleep_charging_ratio, sleep_charging_sum, sleep_charging_transitions, sleep_avg_charging_duration, sleep_max_charging_duration = _process_feature(sleep_status)

    return pd.Series({
        'charging_ratio': charging_ratio,
        'charging_sum': charging_sum,
        'charging_transitions': chargning_transitions,
        'avg_charging_duration': avg_charging_duration,
        'max_charging_duration': max_charging_duration,
        'sleep_charging_ratio': sleep_charging_ratio,
        'sleep_charging_sum': sleep_charging_sum,
        'sleep_charging_transitions': sleep_charging_transitions,
        'sleep_avg_charging_duration': sleep_avg_charging_duration,
        'sleep_max_charging_duration': sleep_max_charging_duration,
    })

mACStatus_ori = load_data(DataType.mACStatus)
mACStatus_ori = shift_lifelog_date(mACStatus_ori, target_hours=SLEEP_HOURS)

mACStatus2  = (
    mACStatus_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mACStatus)
    .reset_index(drop=True)
)

describe_df(mACStatus2)

# shape:
(803, 12)

# dtypes:
subject_id                           category
lifelog_date                   datetime64[ns]
charging_ratio                        float64
charging_sum                          float64
charging_transitions                  float64
avg_charging_duration                 float64
max_charging_duration                 float64
sleep_charging_ratio                  float64
sleep_charging_sum                    float64
sleep_charging_transitions            float64
sleep_avg_charging_duration           float64
sleep_max_charging_duration           float64
dtype: object



,subject_id,lifelog_date,charging_ratio,charging_sum,charging_transitions,avg_charging_duration,max_charging_duration,sleep_charging_ratio,sleep_charging_sum,sleep_charging_transitions,sleep_avg_charging_duration,sleep_max_charging_duration
0,id01,2024-06-26,0.1498,147.0000,22.0000,13.3636,41.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,id01,2024-06-27,0.1650,231.0000,33.0000,13.5882,65.0000,0.0300,9.0000,1.0000,9.0000,9.0000
2,id01,2024-06-28,0.3764,527.0000,28.0000,35.1333,356.0000,1.0000,280.0000,0.0000,280.0000,280.0000


# nan_stats:
                             missing_count  missing_ratio(%)
subject_id                               0            0.0000
lifelog_date                             0            0.0000
charging_ratio                           0            0.0000
charging_sum                             0            0.0000
charging_transitions                     0            0.0000
avg_charging_duration                    0            0.0000
max_charging_duration                    0            0.0000
sleep_charging_ratio                     0            0.0000
sleep_charging_sum                       0            0.0000
sleep_charging_transitions               0            0.0000
sleep_avg_charging_duration              0            0.0000
sleep_max_charging_duration              0            0.0000



### ✔️ mActivity 추정행동
- Value calculated by the Google Activity Recognition API.
 - 0 : IN_VEHICLE
 - 1 : ON_BICYCLE
 - 2 : ON_FOOT
 - 3 : STILL (not moving)
 - 4 : UNKNOWN
 - 5 : TILTING (This often occurs when a device is picked up from a desk or a user who is sitting stands up.)
 - 7 : WALKING
 - 8 : RUNNING
- 근무시간   : 오전 7시부터 오후 6시까지
- 근무외시간 : 오후6시부터 12시까지

In [12]:
def process_mActivity(df):
    activity = df["m_activity"].values.astype("int8")

    EXCLUDE_ACTIVITY = [3, 4]
    WALKING_ACTIVITY = [1, 2, 7, 8]
    VEHICLE_ACTIVITY = [0]

    def _process_feature(activity):
        if len(activity) == 0:
            return 0., 0., 0.

        # Walking minutes
        walking_minutes = np.isin(activity, WALKING_ACTIVITY).sum()

        # Vehicle minutes
        vehicle_minutes = np.isin(activity, VEHICLE_ACTIVITY).sum()

        # Activity minutes
        activity_minutes = (1 - np.isin(activity, EXCLUDE_ACTIVITY)).sum()

        return walking_minutes, vehicle_minutes, activity_minutes

    # 하루
    walking_minutes, vehicle_minutes, activity_minutes = _process_feature(activity)

    # 잠자는 시간대
    sleep_walking_minutes, sleep_vehicle_minutes, sleep_activity_minutes = _process_feature(activity[df["hour"].isin(SLEEP_HOURS)])

    def _metabolic(activity):
        """
        각 활동 코드에 해당하는 MET(Metabolic Equivalent of Task) 값 할당
        MET는 신체 활동의 에너지 소비량을 측정하는 단위

        활동 코드별 MET 값:
            0: 1.3 MET (가벼운 좌식 활동)
            1: 8.0 MET (격렬한 활동)
            3: 1.2 MET (매우 가벼운 활동)
            4: 3.0 MET (중간 강도 활동)
            7: 3.5 MET (중간 강도 활동)
            8: 10.0 MET (매우 격렬한 활동)
        """
        met_values = {
            0: 1.3,  # 가벼운 좌식 활동
            1: 8.0,  # 격렬한 활동
            2: 3.5,  # 중간 강도 활동
            3: 1.2,  # 매우 가벼운 활동
            4: 3.0,  # 중간 강도 활동
            7: 3.5,  # 중간 강도 활동
            8: 10.0, # 매우 격렬한 활동
        }
        mets = np.array([met_values.get(act, 0.) for act in activity])

        # Met 통계
        met_mean = mets.mean()
        met_sum = mets.sum()

        return met_mean, met_sum
    
    # 하루 MET
    met_mean, met_sum = _metabolic(activity)

    return pd.Series({
        'walking_minutes': walking_minutes,
        'vehicle_minutes': vehicle_minutes,
        'activity_minutes': activity_minutes,
        'sleep_walking_minutes': sleep_walking_minutes,
        'sleep_vehicle_minutes': sleep_vehicle_minutes,
        'sleep_activity_minutes': sleep_activity_minutes,
        'met_mean': met_mean,
        'met_sum': met_sum,
    })

mActivity_ori = load_data(DataType.mActivity)
mActivity_ori = shift_lifelog_date(mActivity_ori, target_hours=SLEEP_HOURS)

mActivity2 = (
    mActivity_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mActivity)
    .reset_index(drop=True)
)

describe_df(mActivity2)

# shape:
(803, 10)

# dtypes:
subject_id                      category
lifelog_date              datetime64[ns]
walking_minutes                  float64
vehicle_minutes                  float64
activity_minutes                 float64
sleep_walking_minutes            float64
sleep_vehicle_minutes            float64
sleep_activity_minutes           float64
met_mean                         float64
met_sum                          float64
dtype: object



,subject_id,lifelog_date,walking_minutes,vehicle_minutes,activity_minutes,sleep_walking_minutes,sleep_vehicle_minutes,sleep_activity_minutes,met_mean,met_sum
0,id01,2024-06-26,32.0000,89.0000,121.0000,0.0000,0.0000,0.0000,2.0196,2041.8000
1,id01,2024-06-27,31.0000,211.0000,242.0000,0.0000,0.0000,0.0000,1.2867,1852.8000
2,id01,2024-06-28,37.0000,161.0000,198.0000,0.0000,0.0000,0.0000,1.2747,1835.5000


# nan_stats:
                        missing_count  missing_ratio(%)
subject_id                          0            0.0000
lifelog_date                        0            0.0000
walking_minutes                     0            0.0000
vehicle_minutes                     0            0.0000
activity_minutes                    0            0.0000
sleep_walking_minutes               0            0.0000
sleep_vehicle_minutes               0            0.0000
sleep_activity_minutes              0            0.0000
met_mean                            0            0.0000
met_sum                             0            0.0000



### ✔️ mAmbience 추정주변소리
- Ambient sound identification labels and their respective probabilities.
- 무슨 소리가 난게 중요할까?
- 새벽에 무슨 소리던지 소리가 난게 중요한 걸까?
- 여러 가지 소리 중에 노이즈도 포함되어 있을까?

In [13]:
def process_mAmbience(df):
    ambience = df["m_ambience"].values  # [[label, prob], ...], [[label, prob], ...]

    def _process_feature(ambience):
        labels = set()

        for amb in ambience:
            labels_, _ = zip(*amb)
            labels.update(labels_)

        unique_label_count = len(labels)
        snor_count = len(list(filter(lambda x: "snor" in x.lower(), labels)))

        return unique_label_count, snor_count

    # 활동시간
    active_hour_unique_label_count, active_hour_snor_count = _process_feature(ambience[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는시간
    sleep_hour_unique_label_count, sleep_hour_snor_count = _process_feature(ambience[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_unique_label_count': active_hour_unique_label_count,
        'active_hour_snor_count': active_hour_snor_count,
        'sleep_hour_unique_label_count': sleep_hour_unique_label_count,
        'sleep_hour_snor_count': sleep_hour_snor_count,
    })

mAmbience_ori = load_data(DataType.mAmbience)
mAmbience_ori = shift_lifelog_date(mAmbience_ori, target_hours=SLEEP_HOURS)

mAmbience2 = (
    mAmbience_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mAmbience)
    .reset_index(drop=True)
)

describe_df(mAmbience2)

# shape:
(803, 6)

# dtypes:
subject_id                              category
lifelog_date                      datetime64[ns]
active_hour_unique_label_count             int64
active_hour_snor_count                     int64
sleep_hour_unique_label_count              int64
sleep_hour_snor_count                      int64
dtype: object



,subject_id,lifelog_date,active_hour_unique_label_count,active_hour_snor_count,sleep_hour_unique_label_count,sleep_hour_snor_count
0,id01,2024-06-26,265,2,10,0
1,id01,2024-06-27,10,0,10,0
2,id01,2024-06-28,14,0,10,0


# nan_stats:
                                missing_count  missing_ratio(%)
subject_id                                  0            0.0000
lifelog_date                                0            0.0000
active_hour_unique_label_count              0            0.0000
active_hour_snor_count                      0            0.0000
sleep_hour_unique_label_count               0            0.0000
sleep_hour_snor_count                       0            0.0000



### ✔️ mBle 블루투스
- Bluetooth devices around individual subject.
 - 7936 : Wearable, Headset, AV Device
 - 1796 : Peripheral (입력장치) 계열
 - 0 : 정보 없음 또는 알 수 없음(Unknown)
 - 1084 : Audio/Video (스피커, 헤드셋, 이어폰, TV 등)
 - 524 : Phone (휴대폰, 스마트폰)
 - 1060 : Headphones
 - 284 : commputer (PC, 노트북, PDA)

In [14]:
def process_mBle(df):
    ble = df["m_ble"].values  # [[{"address": "xx:xx:xx:xx:xx:xx", "device_class": "0", "rssi": -70}, ...], [...], ...]

    def _process_feature(ble):
        if len(ble) == 0:
            return 0., 0., 0., 0., 0.

        rssi = []
        devices = []
        for ble_data in ble:
            for device in ble_data:
                rssi.append(device["rssi"])
                devices.append(device["device_class"])

        rssi = np.array(rssi)
        rssi_mean = rssi.mean() if len(rssi) > 0 else 0
        rssi_min = rssi.min() if len(rssi) > 0 else 0
        rssi_max = rssi.max() if len(rssi) > 0 else 0

        unknown_count = devices.count("0")
        others_count = len(devices) - unknown_count
        others_ratio = others_count / len(devices) if len(devices) > 0 else 0
        unknown_ratio = unknown_count / len(devices) if len(devices) > 0 else 0

        return rssi_mean, rssi_min, rssi_max, others_ratio, unknown_ratio

    # 일할때
    work_hour_rssi_mean, work_hour_rssi_min, work_hour_rssi_max, work_hour_others_ratio, work_hour_unknown_ratio = _process_feature(ble[df["hour"].isin(WORK_HOURS)])

    # 퇴근후
    free_hour_rssi_mean, free_hour_rssi_min, free_hour_rssi_max, free_hour_others_ratio, free_hour_unknown_ratio = _process_feature(ble[df["hour"].isin(FREE_HOURS)])

    # 잠자는시간
    sleep_hour_rssi_mean, sleep_hour_rssi_min, sleep_hour_rssi_max, sleep_hour_others_ratio, sleep_hour_unknown_ratio = _process_feature(ble[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'work_hour_rssi_mean': work_hour_rssi_mean,
        'work_hour_rssi_min': work_hour_rssi_min,
        'work_hour_rssi_max': work_hour_rssi_max,
        'work_hour_others_ratio': work_hour_others_ratio,
        'work_hour_unknown_ratio': work_hour_unknown_ratio,
        'free_hour_rssi_mean': free_hour_rssi_mean,
        'free_hour_rssi_min': free_hour_rssi_min,
        'free_hour_rssi_max': free_hour_rssi_max,
        'free_hour_others_ratio': free_hour_others_ratio,
        'free_hour_unknown_ratio': free_hour_unknown_ratio,
        'sleep_hour_rssi_mean': sleep_hour_rssi_mean,
        'sleep_hour_rssi_min': sleep_hour_rssi_min,
        'sleep_hour_rssi_max': sleep_hour_rssi_max,
        'sleep_hour_others_ratio': sleep_hour_others_ratio,
        'sleep_hour_unknown_ratio': sleep_hour_unknown_ratio
    })

mBle_ori = load_data(DataType.mBle)
mBle_ori = shift_lifelog_date(mBle_ori, target_hours=SLEEP_HOURS)

mBle2 = (
    mBle_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mBle)
    .reset_index(drop=True)
)

describe_df(mBle2)

# shape:
(709, 17)

# dtypes:
subject_id                        category
lifelog_date                datetime64[ns]
work_hour_rssi_mean                float64
work_hour_rssi_min                 float64
work_hour_rssi_max                 float64
work_hour_others_ratio             float64
work_hour_unknown_ratio            float64
free_hour_rssi_mean                float64
free_hour_rssi_min                 float64
free_hour_rssi_max                 float64
free_hour_others_ratio             float64
free_hour_unknown_ratio            float64
sleep_hour_rssi_mean               float64
sleep_hour_rssi_min                float64
sleep_hour_rssi_max                float64
sleep_hour_others_ratio            float64
sleep_hour_unknown_ratio           float64
dtype: object



,subject_id,lifelog_date,work_hour_rssi_mean,work_hour_rssi_min,work_hour_rssi_max,work_hour_others_ratio,work_hour_unknown_ratio,free_hour_rssi_mean,free_hour_rssi_min,free_hour_rssi_max,free_hour_others_ratio,free_hour_unknown_ratio,sleep_hour_rssi_mean,sleep_hour_rssi_min,sleep_hour_rssi_max,sleep_hour_others_ratio,sleep_hour_unknown_ratio
0,id01,2024-06-26,-74.0904,-94.0000,-27.0000,0.0590,0.9410,-77.2213,-92.0000,-43.0000,0.0791,0.9209,0.0000,0.0000,0.0000,0.0000,0.0000
1,id01,2024-06-27,-73.7473,-94.0000,-34.0000,0.0614,0.9386,-74.6667,-91.0000,-42.0000,0.1167,0.8833,0.0000,0.0000,0.0000,0.0000,0.0000
2,id01,2024-06-28,-75.7993,-92.0000,-39.0000,0.0467,0.9533,-77.2558,-94.0000,-51.0000,0.3256,0.6744,0.0000,0.0000,0.0000,0.0000,0.0000


# nan_stats:
                          missing_count  missing_ratio(%)
subject_id                            0            0.0000
lifelog_date                          0            0.0000
work_hour_rssi_mean                   0            0.0000
work_hour_rssi_min                    0            0.0000
work_hour_rssi_max                    0            0.0000
work_hour_others_ratio                0            0.0000
work_hour_unknown_ratio               0            0.0000
free_hour_rssi_mean                   0            0.0000
free_hour_rssi_min                    0            0.0000
free_hour_rssi_max                    0            0.0000
free_hour_others_ratio                0            0.0000
free_hour_unknown_ratio               0            0.0000
sleep_hour_rssi_mean                  0            0.0000
sleep_hour_rssi_min                   0            0.0000
sleep_hour_rssi_max                   0            0.0000
sleep_hour_others_ratio               0            0.0000
s

### ✔️ mGps, GPS 기반 핸드폰 위치
- Multiple GPS coordinates measured within a single minute using the smartphone.
- speed가 1보다 큰경우 정지 상태가 아니고 움직이고 있다고 판단
 - 0.5-2 : 걸어서 이동하는 경우  
 - 2-5 : 조깅
 - 5 이상 : 차를 타고 이동하는 경우

- speed가 0.5-2사이를 하루에 몇분동안 지속했는지?
- speed가 2-5사이를 하루에 몇분동안 지속했는지? (유산소 운동 시간)
- speed가 5이상을 하루에 몇분동안 지속했는지?

In [15]:
from datetime import datetime

def haversine_np(lon1, lat1, lon2, lat2, radius=6371):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return radius * c

def process_mGps(df):
    gps = df["m_gps"].values  # [[{'altitude': 110.6, 'latitude': 0.2077385, 'longitude': 0.170027, 'speed': 0.0}, ...], ...]
    timestamps = df["timestamp"].values

    def _process_feature(gps, timestamps):
        if len(gps) == 0:
            return 0., 0., 0., 0., 0., 0., 0., np.array([])

        # n-분 단위
        latitudes = []
        longitudes = []
        altitudes = []
        speeds = []
        minutes = []  # 누적 분

        for i, (gps_data, timestamp) in enumerate(zip(gps, timestamps)):
            _latitudes = []
            _longitudes = []
            _altitudes = []
            _speeds = []
            for data in gps_data:
                _latitudes.append(data["latitude"])
                _longitudes.append(data["longitude"])
                _altitudes.append(data["altitude"])
                _speeds.append(data["speed"])

            latitudes.append(np.mean(_latitudes))
            longitudes.append(np.mean(_longitudes))
            altitudes.append(np.mean(_altitudes))
            speeds.append(np.mean(_speeds))
            minutes.append(1 if i == 0 else pd.Timedelta(timestamps[i] - timestamps[i-1]).total_seconds() / 60)

        latitudes = np.array(latitudes)
        longitudes = np.array(longitudes)
        altitudes = np.array(altitudes)
        speeds = np.array(speeds)
        minutes = np.array(minutes)

        walk_minutes = minutes[(speeds >= 0.5) & (speeds < 2.0)].sum()
        jog_minutes = minutes[(2.0 <= speeds) & (speeds < 5.0)].sum()
        vehicle_minutes = minutes[(5.0 <= speeds)].sum()

        # 속도
        mean_speed = speeds.mean() if len(speeds) > 0 else 0
        max_speed = speeds.max() if len(speeds) > 0 else 0
        min_speed = speeds.min() if len(speeds) > 0 else 0

        # 이동거리
        distance = haversine_np(longitudes[:-1], latitudes[:-1], longitudes[1:], latitudes[1:]).sum()

        return walk_minutes, jog_minutes, vehicle_minutes, mean_speed, max_speed, min_speed, distance, speeds

    # 하루
    active_hour_walk_minutes, active_hour_jog_minutes, active_hour_vehicle_minutes, active_hour_mean_speed, active_hour_max_speed, active_hour_min_speed, active_hour_distance, _ = _process_feature(gps[df["hour"].isin(ACTIVE_HOURS)], timestamps[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_walk_minutes, sleep_hour_jog_minutes, sleep_hour_vehicle_minutes, sleep_hour_mean_speed, sleep_hour_max_speed, sleep_hour_min_speed, sleep_hour_distance, _ = _process_feature(gps[df["hour"].isin(SLEEP_HOURS)], timestamps[df["hour"].isin(SLEEP_HOURS)])

    # 일어날 때
    _, _, _, _, _, _, _, might_wakeup_speeds = _process_feature(gps[df["hour"].isin(MIGHT_WAKEUP_HOURS)], timestamps[df["hour"].isin(MIGHT_WAKEUP_HOURS)])
    might_wakeup_timestamps = timestamps[df["hour"].isin(MIGHT_WAKEUP_HOURS)]
    wakeup_timestamps = might_wakeup_timestamps[(might_wakeup_speeds > 1.0)]
    first_move_datetime = (
        pd.to_datetime(wakeup_timestamps[0]) if len(wakeup_timestamps) > 0
        else pd.to_datetime(might_wakeup_timestamps[-1]) if len(might_wakeup_timestamps) > 0
        else pd.to_datetime(datetime(2024, 1, 1, MIGHT_WAKEUP_HOURS[-1], 0, 0))  # default to the last hour of the range
    )
    first_wakeup_minutes = (first_move_datetime.hour if first_move_datetime.hour > 12 else first_move_datetime.hour + 24) * 60 + first_move_datetime.minute

    return pd.Series({
        'active_hour_walk_minutes': active_hour_walk_minutes,
        'active_hour_jog_minutes': active_hour_jog_minutes,
        'active_hour_vehicle_minutes': active_hour_vehicle_minutes,
        'active_hour_mean_speed': active_hour_mean_speed,
        'active_hour_max_speed': active_hour_max_speed,
        'active_hour_min_speed': active_hour_min_speed,
        'active_hour_distance': active_hour_distance,
        'exercise_flag': 1 if active_hour_jog_minutes > 10 else 0,  # n분 이상 조깅한 경우
        'sleep_hour_walk_minutes': sleep_hour_walk_minutes,
        'sleep_hour_jog_minutes': sleep_hour_jog_minutes,
        'sleep_hour_vehicle_minutes': sleep_hour_vehicle_minutes,
        'sleep_hour_mean_speed': sleep_hour_mean_speed,
        'sleep_hour_max_speed': sleep_hour_max_speed,
        'sleep_hour_min_speed': sleep_hour_min_speed,
        'sleep_hour_distance': sleep_hour_distance,
        "mgps_first_wakeup_minutes": first_wakeup_minutes,
    })


mGps_ori = load_data(DataType.mGps)
mGps_ori = shift_lifelog_date(mGps_ori, target_hours=SLEEP_HOURS)

mGps2 = (
    mGps_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mGps)
    .reset_index(drop=True)
)

describe_df(mGps2)

# shape:
(759, 18)

# dtypes:
subject_id                           category
lifelog_date                   datetime64[ns]
active_hour_walk_minutes              float64
active_hour_jog_minutes               float64
active_hour_vehicle_minutes           float64
active_hour_mean_speed                float64
active_hour_max_speed                 float64
active_hour_min_speed                 float64
active_hour_distance                  float64
exercise_flag                         float64
sleep_hour_walk_minutes               float64
sleep_hour_jog_minutes                float64
sleep_hour_vehicle_minutes            float64
sleep_hour_mean_speed                 float64
sleep_hour_max_speed                  float64
sleep_hour_min_speed                  float64
sleep_hour_distance                   float64
mgps_first_wakeup_minutes             float64
dtype: object



,subject_id,lifelog_date,active_hour_walk_minutes,active_hour_jog_minutes,active_hour_vehicle_minutes,active_hour_mean_speed,active_hour_max_speed,active_hour_min_speed,active_hour_distance,exercise_flag,sleep_hour_walk_minutes,sleep_hour_jog_minutes,sleep_hour_vehicle_minutes,sleep_hour_mean_speed,sleep_hour_max_speed,sleep_hour_min_speed,sleep_hour_distance,mgps_first_wakeup_minutes
0,id01,2024-06-26,68.0000,32.0000,19.0000,0.5775,19.0505,0.0000,16.7900,1.0000,36.0000,0.0000,0.0000,0.1812,1.6664,0.0000,0.1752,1980.0000
1,id01,2024-06-27,136.0000,61.0000,66.0000,1.0368,24.2032,0.0000,32.2769,1.0000,1.0000,0.0000,0.0000,0.0424,0.5080,0.0000,0.0647,1831.0000
2,id01,2024-06-28,106.0000,68.0000,42.0000,0.8060,24.1712,0.0001,35.5108,1.0000,19.0000,0.0000,0.0000,0.3053,0.8415,0.0987,0.4590,1834.0000


# nan_stats:
                             missing_count  missing_ratio(%)
subject_id                               0            0.0000
lifelog_date                             0            0.0000
active_hour_walk_minutes                 0            0.0000
active_hour_jog_minutes                  0            0.0000
active_hour_vehicle_minutes              0            0.0000
active_hour_mean_speed                   0            0.0000
active_hour_max_speed                    0            0.0000
active_hour_min_speed                    0            0.0000
active_hour_distance                     0            0.0000
exercise_flag                            0            0.0000
sleep_hour_walk_minutes                  0            0.0000
sleep_hour_jog_minutes                   0            0.0000
sleep_hour_vehicle_minutes               0            0.0000
sleep_hour_mean_speed                    0            0.0000
sleep_hour_max_speed                     0            0.0000
sleep_hour_

### ✔️ mLight 주변 밝기
- Ambient light measured by the smartphone.
 - 어두운 밤	0.1 ~ 1 lux	캄캄한 방, 달빛 없는 밤
 - 가로등 켜진 거리	10 ~ 20 lux	흐릿한 외부 조명
 - 실내 조명	100 ~ 500 lux	사무실, 일반 거실
 - 밝은 실외	10,000 ~ 25,000 lux	맑은 날 햇빛
 - 직사광선 아래	30,000 ~ 100,000 lux	여름 한낮, 매우 강한 햇빛

- 밝기에 따라서 언제 불을 끄고 잠든 시간 추정
- 직사광선 잠에 좋은 영향을 주는지? (논문)
- 결측치 처리 x

In [16]:
def process_mLight(df):
    light = df["m_light"].values  # [534.0, 224, ...]

    def _process_feature(light):
        if len(light) == 0:
            return 0., 0., 0., 0., np.array([])

        ligths = np.array(light)
        mean_light = ligths.mean() if len(ligths) > 0 else 0
        min_light = ligths.min() if len(ligths) > 0 else 0
        max_light = ligths.max() if len(ligths) > 0 else 0
        std_light = ligths.std() if len(ligths) > 0 else 0

        return mean_light, min_light, max_light, std_light, ligths
    
    # 하루
    active_hour_mean_light, active_hour_min_light, active_hour_max_light, active_hour_std_light, _ = _process_feature(light[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_mean_light, sleep_hour_min_light, sleep_hour_max_light, sleep_hour_std_light, _= _process_feature(light[df["hour"].isin(SLEEP_HOURS)])

    # 잠 자러갈 때
    might_go_to_sleep_light = light[df["hour"].isin(MIGHT_GO_TO_SLEEP_HOURS)]
    might_go_to_sleep_timestamps = df["timestamp"].values[df["hour"].isin(MIGHT_GO_TO_SLEEP_HOURS)]
    _, _, _, _, might_go_to_sleep_lights = _process_feature(might_go_to_sleep_light)
    first_sleep_timestamps = might_go_to_sleep_timestamps[(might_go_to_sleep_lights < 10.0)]
    first_sleep_datetime = (
        pd.to_datetime(first_sleep_timestamps[0]) if len(first_sleep_timestamps) > 0 
        else pd.to_datetime(might_go_to_sleep_timestamps[-1]) if len(might_go_to_sleep_timestamps) > 0
        else pd.to_datetime(datetime(2024, 1, 1, MIGHT_GO_TO_SLEEP_HOURS[-1], 0, 0))  # default to the last hour of the range
    )
    first_sleep_minutes = datetime_to_minute(first_sleep_datetime)

    # 일어날 때
    might_wakeup_light = light[df["hour"].isin(MIGHT_WAKEUP_HOURS)]
    might_wakeup_timestamps = df["timestamp"].values[df["hour"].isin(MIGHT_WAKEUP_HOURS)]
    _, _, _, _, might_wakeup_lights = _process_feature(might_wakeup_light)
    wakeup_timestamps = might_wakeup_timestamps[(might_wakeup_lights > 10.0)]
    first_move_datetime = (
        pd.to_datetime(wakeup_timestamps[0]) if len(wakeup_timestamps) > 0 
        else pd.to_datetime(might_wakeup_timestamps[-1]) if len(might_wakeup_timestamps) > 0
        else pd.to_datetime(datetime(2024, 1, 1, MIGHT_WAKEUP_HOURS[-1], 0, 0))  # default to the last hour of the range
    )
    first_wakeup_minutes = datetime_to_minute(first_move_datetime)

    mlight_sleep_duration = first_wakeup_minutes - first_sleep_minutes if first_wakeup_minutes > first_sleep_minutes else 0

    # SLEEP_HOURS 동안 상태 전이 횟수
    sleep_light = light[df["hour"].isin(SLEEP_HOURS)]
    if len(sleep_light) == 0:
        sleep_light_transitions = 0
    else:
        # 상태 전이 계산
        sleep_light_diff = np.diff(sleep_light, prepend=sleep_light[0])
        sleep_light_transitions = np.sum(np.abs(sleep_light_diff))

    return pd.Series({
        'active_hour_mean_light': active_hour_mean_light,
        'active_hour_min_light': active_hour_min_light,
        'active_hour_max_light': active_hour_max_light,
        'active_hour_std_light': active_hour_std_light,
        'sleep_hour_mean_light': sleep_hour_mean_light,
        'sleep_hour_min_light': sleep_hour_min_light,
        'sleep_hour_max_light': sleep_hour_max_light,
        'sleep_hour_std_light': sleep_hour_std_light,
        'mlight_first_sleep_minutes': first_sleep_minutes,
        'mlight_first_wakeup_minutes': first_wakeup_minutes,
        'mlight_sleep_duration': mlight_sleep_duration,
        'mlight_sleep_transitions': sleep_light_transitions,
    })

mLight_ori = load_data(DataType.mLight)
mLight_ori = shift_lifelog_date(mLight_ori, target_hours=SLEEP_HOURS)

mLight2 = (
    mLight_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mLight)
    .reset_index(drop=True)
)
describe_df(mLight2)

# shape:
(802, 14)

# dtypes:
subject_id                           category
lifelog_date                   datetime64[ns]
active_hour_mean_light                float64
active_hour_min_light                 float64
active_hour_max_light                 float64
active_hour_std_light                 float64
sleep_hour_mean_light                 float64
sleep_hour_min_light                  float64
sleep_hour_max_light                  float64
sleep_hour_std_light                  float64
mlight_first_sleep_minutes            float64
mlight_first_wakeup_minutes           float64
mlight_sleep_duration                 float64
mlight_sleep_transitions              float64
dtype: object



,subject_id,lifelog_date,active_hour_mean_light,active_hour_min_light,active_hour_max_light,active_hour_std_light,sleep_hour_mean_light,sleep_hour_min_light,sleep_hour_max_light,sleep_hour_std_light,mlight_first_sleep_minutes,mlight_first_wakeup_minutes,mlight_sleep_duration,mlight_sleep_transitions
0,id01,2024-06-26,364.5068,0.0000,1886.0000,392.9401,0.0000,0.0000,0.0000,0.0000,1203.0000,1980.0000,777.0000,0.0000
1,id01,2024-06-27,450.5784,0.0000,11248.0000,1521.9885,0.0000,0.0000,0.0000,0.0000,1209.0000,1839.0000,630.0000,0.0000
2,id01,2024-06-28,291.2941,0.0000,1834.0000,267.0134,0.0000,0.0000,0.0000,0.0000,1389.0000,1839.0000,450.0000,0.0000


# nan_stats:
                             missing_count  missing_ratio(%)
subject_id                               0            0.0000
lifelog_date                             0            0.0000
active_hour_mean_light                   0            0.0000
active_hour_min_light                    0            0.0000
active_hour_max_light                    0            0.0000
active_hour_std_light                    0            0.0000
sleep_hour_mean_light                    0            0.0000
sleep_hour_min_light                     0            0.0000
sleep_hour_max_light                     0            0.0000
sleep_hour_std_light                     0            0.0000
mlight_first_sleep_minutes               0            0.0000
mlight_first_wakeup_minutes              0            0.0000
mlight_sleep_duration                    0            0.0000
mlight_sleep_transitions                 0            0.0000



### 🔥 mScreenStatus 화면 사용여부

- Indicates whether the smartphone screen is in use.
 - 기상시간, 취침시간, 수면시간
 - 휴대폰 이용횟수, 이용시간
 - 00 - 05 사이에 휴대폰 이용한 건수
 - 결측치 처리 x

In [17]:
def process_mScreenStatus(df):
    screen_use = df["m_screen_use"].values  # [0, 1, 0, ...]
    screen_use = np.array(screen_use).astype("int8")

    def _process_feature(screen_use):
        if len(screen_use) == 0:
            return 0., 0., 0.

        screen_uses = np.array(screen_use)
        screen_use_ratio = screen_uses.mean() if len(screen_uses) > 0 else 0
        screen_use_sum = screen_uses.sum() if len(screen_uses) > 0 else 0
        screen_use_transitions = (screen_uses[1:] != screen_uses[:-1]).sum()

        return screen_use_ratio, screen_use_sum, screen_use_transitions
    
    # 하루
    screen_use_ratio, screen_use_sum, screen_use_transitions = _process_feature(screen_use)

    # 잠자는 시간대
    sleep_screen_use_ratio, sleep_screen_use_sum, sleep_screen_use_transitions = _process_feature(screen_use[df["hour"].isin(SLEEP_HOURS)])

    # 잠 잔 시간, 일어난 시간 (window 사용)
    # 화면 미사용(0)이 일정 시간(window) 이상 지속되면 '수면'으로 간주
    window = 3  # 분 단위 window, 필요시 조정
    screen_use_series = pd.Series(screen_use[df["hour"].isin(MIGHT_GO_TO_SLEEP_HOURS)])
    screen_use_series = screen_use_series.rolling(window=window, min_periods=1).sum()
    first_sleep_timestamps = screen_use_series[screen_use_series == 0].index
    first_sleep_datetime = (
        pd.to_datetime(first_sleep_timestamps[0]) if len(first_sleep_timestamps) > 0
        else pd.to_datetime(df["timestamp"].values[df["hour"].isin(MIGHT_GO_TO_SLEEP_HOURS)][-1]) if len(df["timestamp"].values[df["hour"].isin(MIGHT_GO_TO_SLEEP_HOURS)]) > 0
        else pd.to_datetime(datetime(2024, 1, 1, MIGHT_GO_TO_SLEEP_HOURS[-1], 0, 0))  # default to the last hour of the range
    )
    first_sleep_minutes = datetime_to_minute(first_sleep_datetime)

    screen_use_series = pd.Series(screen_use[df["hour"].isin(MIGHT_WAKEUP_HOURS)])
    screen_use_series = screen_use_series.rolling(window=window, min_periods=1).sum()
    first_wakeup_timestamps = screen_use_series[screen_use_series == 1].index
    first_wakeup_datetime = (
        pd.to_datetime(first_wakeup_timestamps[0]) if len(first_wakeup_timestamps) > 0
        else pd.to_datetime(df["timestamp"].values[df["hour"].isin(MIGHT_WAKEUP_HOURS)][-1]) if len(df["timestamp"].values[df["hour"].isin(MIGHT_WAKEUP_HOURS)]) > 0
        else pd.to_datetime(datetime(2024, 1, 1, MIGHT_WAKEUP_HOURS[-1], 0, 0))  # default to the last hour of the range
    )
    first_wakeup_minutes = datetime_to_minute(first_wakeup_datetime)

    mscreen_sleep_duration = first_wakeup_minutes - first_sleep_minutes if first_wakeup_minutes > first_sleep_minutes else 0

    return pd.Series({
        'screen_use_ratio': screen_use_ratio,
        'screen_use_sum': screen_use_sum,
        'screen_use_transitions': screen_use_transitions,
        'sleep_screen_use_ratio': sleep_screen_use_ratio,
        'sleep_screen_use_sum': sleep_screen_use_sum,
        'sleep_screen_use_transitions': sleep_screen_use_transitions,
        'mscreen_first_sleep_minutes': first_sleep_minutes,
        'mscreen_first_wakeup_minutes': first_wakeup_minutes,
        'mscreen_sleep_duration': mscreen_sleep_duration,
    })


mScreenStatus_ori = load_data(DataType.mScreenStatus)
mScreenStatus_ori = shift_lifelog_date(mScreenStatus_ori, target_hours=SLEEP_HOURS)

mScreenStatus2 = (
    mScreenStatus_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mScreenStatus)
    .reset_index(drop=True)
)
describe_df(mScreenStatus2)


# shape:
(803, 11)

# dtypes:
subject_id                            category
lifelog_date                    datetime64[ns]
screen_use_ratio                       float64
screen_use_sum                         float64
screen_use_transitions                 float64
sleep_screen_use_ratio                 float64
sleep_screen_use_sum                   float64
sleep_screen_use_transitions           float64
mscreen_first_sleep_minutes            float64
mscreen_first_wakeup_minutes           float64
mscreen_sleep_duration                 float64
dtype: object



,subject_id,lifelog_date,screen_use_ratio,screen_use_sum,screen_use_transitions,sleep_screen_use_ratio,sleep_screen_use_sum,sleep_screen_use_transitions,mscreen_first_sleep_minutes,mscreen_first_wakeup_minutes,mscreen_sleep_duration
0,id01,2024-06-26,0.2098,210.0000,54.0000,0.0000,0.0000,0.0000,1440.0000,1980.0000,540.0000
1,id01,2024-06-27,0.3657,523.0000,82.0000,0.0000,0.0000,0.0000,1440.0000,1440.0000,0.0000
2,id01,2024-06-28,0.3175,454.0000,90.0000,0.0000,0.0000,0.0000,1440.0000,1440.0000,0.0000


# nan_stats:
                              missing_count  missing_ratio(%)
subject_id                                0            0.0000
lifelog_date                              0            0.0000
screen_use_ratio                          0            0.0000
screen_use_sum                            0            0.0000
screen_use_transitions                    0            0.0000
sleep_screen_use_ratio                    0            0.0000
sleep_screen_use_sum                      0            0.0000
sleep_screen_use_transitions              0            0.0000
mscreen_first_sleep_minutes               0            0.0000
mscreen_first_wakeup_minutes              0            0.0000
mscreen_sleep_duration                    0            0.0000



### ✔️ mUsageStats 앱사용통계
- mUsageStats: Indicates which apps were used on the smartphone and for how long.

 - 몇시까지 핸드폰 보다가 잠잤는지
 - 통화, 전화 얼마나 했는지
 - YouTube 얼마나 봤는지
 - 메시지, 카카오톡 얼마나 했는지
 - NAVER 얼마나 했는지
 - 평소보다 얼마나 많은 앱을 이용했는지
 - 제외? -> 시스템 UI,One UI 홈

In [18]:
mUsageStats_ori = load_data(DataType.mUsageStats)
mUsageStats_ori = shift_lifelog_date(mUsageStats_ori, target_hours=SLEEP_HOURS)
len(mUsageStats_ori), mUsageStats_ori.head(1)

(45197,
   subject_id           timestamp lifelog_date  month  day  hour  minute  \
 0       id01 2024-06-26 13:00:00   2024-06-26      6   26    13       0   
 
    weekday  \
 0        2   
 
                                                                                                                        m_usage_stats  
 0  [{'app_name': ' 캐시워크', 'total_time': 69}, {'app_name': 'NAVER', 'total_time': 549}, {'app_name': ' ✝️성경일독Q', 'total_time': 7337}]  )

In [19]:
app_names = set()
for app_list in mUsageStats_ori["m_usage_stats"].values:
    for app in app_list:
        app_names.add(app["app_name"])

app_names = sorted(list(app_names))
app_names[:3]

['(구)티머니onda', '(캐시아워)', '11번가']

In [20]:
# gpt 한테 카테고리 만들라고 시킴 (https://chatgpt.com/share/e/6825e3d0-4bc0-8009-98ad-ee3836bc0fd3)
app_category_to_names = { "금융": [ "11번가", "AIA생명", "IBK 기업은행", "KB Pay", "KB스타뱅킹", "KS Fit", "MG더뱅킹", "MG손해보험 다이렉트", "NH pay", "NH기업뱅킹", "NH뱅킹", "NH앱캐시", "NH콕뱅크", "OK Cashbag", "PASS", "PAYCO", "Samsung Wallet", "Syrup", "The건강보험", "Toss", "핀크", "하나머니", "하나은행", "하나카드", "한국투자", "한화손해보험", "현대카드", "흥국화재", "카카오페이", "케이뱅크", "토스", "우체국보험", "웰컴디지털뱅크", "카카오뱅크", "삼성카드", "삼성화재 다이렉트 착" ], "기타": [ "(구)티머니onda", "(캐시아워)", "AirVisual", "AlwaysOnDisplay", "Android 시스템", "AnyAUTH", "Arkcraft_v1", "Authentication Framework", "Auto Clicker", "Auto Redial", "Avis Corporate", "BNKR몰", "CHARGEV", "CNCITY에너지", "DRAWELY", "Expert RAW", "Gaming Hub", "Gateman", "Good Lock", "Google", "Google 음성 인식 및 합성", "H.Point", "Headphones", "Home", "IDF Mobilités", "IntentResolver", "K-패스", "KAIST IdCard", "LAVU", "LG ThinQ", "LH청약센터", "LIVE스코어", "Letter Fonts", "MTP 애플리케이션", "MY FANS", "MY네컷", "Maps & Navigation", "Master for Minecraft", "Mi Home", "Mobile HR", "MyKia", "NFC", "One UI 홈", "Pi", "Prime Ruler", "Quick Share", "RoomEstimateApp", "SIM 카드 툴킷", "Samsung Checkout", "Samsung Pass 자동 완성", "SecSoundPicker", "Secure SignIn", "Smart Home", "Smart Switch", "Smart View", "Smart​Things", "Start", "StudioMate", "Tasks", "T world", "TD infinite", "Tapo", "TasteBuds", "VpnDialogs", "WORKS", "Whiteout Survival", "Wi-Fi 연결 팁", "Windows와 연결", "Yodha Pro", "b.stage", "blind", "com.dreamsecurity.MobileRelay.SampleCrypto", "duit+", "help-CNUH24", "i-ONE 알림", "iM뱅크", "monimo", "vFlat Scan", "가족돌봄", "계산기", "고용24", "교육원 전자출결", "국세청 손택스", "굿웨어몰", "권한 관리자", "긴급 SOS", "데이터 복원 도구", "디바이스 케어", "디지털 웰빙", "땡큐캠핑", "똑똑계산기", "멀티 컨트롤", "미디어 선택 도구", "바로청구", "배경화면 및 스타일", "보안 폴더", "부속 기기 관리자", "비디오 플레이어", "빅스비 루틴", "빅스비 보이스", "빅스비 비전", "삼성 계정", "삼성 캡처", "삼성 클라우드", "삼성 키보드", "생체 인식", "설정", "시계", "시스템 UI", "시프티", "앱 소리 분리 재생", "어시스턴트", "연락처", "영상통화 효과", "오피넷", "주변 디바이스 찾기", "접근성", "추천 설정", "추천 앱", "출입예약시스템", "키 체인", "통화", "통화 설정", "패키지 설치 프로그램", "휴대폰분실보호", "히어로즈" ], "쇼핑": [ "11번가", "ABC-MART", "AliExpress", "Amway", "BHC", "BNKR몰", "CJ온스타일", "GS SHOP", "G마켓", "KREAM", "KT알파 쇼핑", "NS홈쇼핑", "Nike", "SHEIN", "SSG.COM", "Temu", "iHerb", "무신사", "다나와 가격비교", "다이소몰", "롯데ON", "롯데잇츠", "롯데홈쇼핑", "메가MGC커피", "메가박스", "버거킹", "보리보리", "신세계몰", "신세계쇼핑", "아이디어스", "에누리 가격비교", "에이블리", "엘포인트", "옥션", "올리브영", "컴포즈커피", "쿠우쿠우", "쿠쿠통합몰", "쿠팡", "쿠팡이츠", "쿠팡플레이", "퀸잇", "크몽", "하이버", "하프클럽", "홈플러스", "이마트24", "제주항공", "젤캔들샵", "카지노" ], "게임": [ "1945 Air Force", "Arkcraft_v1", "Art Puzzle", "Block Journey", "Block Puzzle", "Cake Sort", "Crossy Road", "Cytus II", "Darkness and Flame 1", "Darkness and Flame 2", "Darkness and Flame 3", "Darkness and Flame 4", "Domino Dreams", "FC Online M", "Fantastic Bricks", "Find Differences", "Find Out", "Friends Rush", "Goblins Wood", "Killer Sudoku", "Magic Tiles 3", "Merge Designer-Decor & Story", "Nonogram Elf", "Nonogram-Number games", "NumMatch", "Number Match", "Nuts & Bolts Jam", "Plague Inc.", "PokeRogue Offline", "Pokémon GO", "Pokémon UNITE", "Royal Match", "SimCity", "Steam", "Super Slime - Black Hole Game", "TFT", "Talking Tom Gold Run", "Whiteout Survival", "WoW 컴패니언", "눈을 떠요 야생소년", "매직 디펜스", "무한의 계단", "벽돌깨기 퀘스트", "세븐나이츠", "수확의 정석", "카트라이더 러쉬플러스", "클래시 로얄", "클래시오브클랜", "픽셀 포켓 모험", "퓨처파이트", "프렌즈팝콘" ], "여행/교통": [ "Android Auto", "Avenza Maps", "Booking.com", "Flightradar24", "Google Maps", "Grab", "Hotels.com", "IDF Mobilités", "Kia", "Kia Connect", "Kia Digital Key", "Omio", "Skyscanner", "TMAP", "Trip.com", "U+스마트홈", "Waze", "고속버스 티머니", "대전버스", "대전시 타슈(QR단말기전용)", "카카오내비", "카카오맵", "카카오버스", "코레일톡", "티머니GO", "티웨이항공" ], "건강": [ "GoFasting", "InBody", "KS Fit", "Samsung Health", "Withings" ], "음악": [ "FLO", "Melon", "YouTube Music", "무료음악 벨소리", "지니뮤직" ], "사진/영상": [ "AI Retouch - 객체 제거", "AR 이모지", "AR 이모지 스티커", "AR 이모지 에디터", "AR 존", "Adobe Acrobat", "B612", "CamScanner", "SNOW", "YouTube", "넷﻿플﻿릭﻿스", "TikTok", "TikTok-Lite" ], "소셜": [ "BAND", "Facebook", "Instagram", "LINE", "LIVE스코어", "Messenger", "Slack", "StarMaker", "Threads", "WeChat", "X", "blind", "리멤버" ], "생산성": [ "Drive", "Files by Google", "Gmail", "Microsoft 365 (Office)", "Microsoft Word", "Notion", "Outlook", "PowerPoint", "Samsung Notes", "TimeTree", "WPS Office", "Zoom", "네이버 MYBOX", "스프레드시트", "프레젠테이션" ], "유틸리티": [ "AhnLab V3 Mobile Plus", "AlwaysOnDisplay", "AnyAUTH", "Authenticator", "Edge", "ExpressVPN", "Files by Google", "Google Play 서비스", "Google Play 스토어", "Home", "Microsoft Launcher", "One UI 홈", "Quick Share", "Samsung Flow", "Samsung Free", "Samsung Members", "Samsung Pass", "Smart View" ], "교육": [ "갓피플성경", "개역개정 큰글성경", "성경", "항전ON", "합격 요양보호사" ], "식음료": [ "McDonald's", "SRT", "Starbucks", "배달의민족", "요기요", "엽기떡볶이", "투썸하트" ], "뉴스/정보": [ "Flightradar24", "네이트", "네이트메일", "재난문자" ], "라이프스타일": [ "Android 시스템", "LG ThinQ", "오늘의집", "헤이홈", "홈노크타운", "숲나들e", "지그재그", "핫핑", "샐러디", "아이쉐어링", "와디즈", "포인핸드" ] }
app_categories = list(app_category_to_names.keys())
app_name_to_category = {}
for category, names in app_category_to_names.items():
    for name in names:
        app_name_to_category[name] = category

In [21]:
def process_mUsageStats(df):
    usage_stats = df["m_usage_stats"].values  #  [[{'app_name': ' 캐시워크', 'total_time': 69}, {'app_name': 'NAVER', 'total_time': 549}, {'app_name': ' ✝️성경일독Q', 'total_time': 7337}], [...], ...] 

    def _process_feature(usage_stats):
        # 앱 카테고리별 사용시간
        category_usage_times = {
            category: 0. for category in app_categories
        }
        category_usage_times["UNKNOWN"] = 0.

        if len(usage_stats) == 0:
            return category_usage_times

        for usage_stat in usage_stats:
            for app in usage_stat:
                app_name = app["app_name"]
                useage_time = app["total_time"] * 0.001 / 60  # 밀리초 -> 초 -> 분

                category_usage_times[app_name_to_category.get(app_name, "UNKNOWN")] += useage_time
        
        return category_usage_times

    # 하루
    active_hour_category_usage_times = _process_feature(usage_stats[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_category_usage_times = _process_feature(usage_stats[df["hour"].isin(SLEEP_HOURS)])

    ret = {
        **{
            f"active_hour_{category}_usage_time": active_hour_category_usage_times[category]
            for category in app_categories
        },
        **{
            f"sleep_hour_{category}_usage_time": sleep_hour_category_usage_times[category]
            for category in app_categories
        }
    }

    return pd.Series(ret)

mUsageStats2 = (
    mUsageStats_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mUsageStats)
    .reset_index(drop=True)
)
describe_df(mUsageStats2)


# shape:
(784, 32)

# dtypes:
subject_id                             category
lifelog_date                     datetime64[ns]
active_hour_금융_usage_time               float64
active_hour_기타_usage_time               float64
active_hour_쇼핑_usage_time               float64
active_hour_게임_usage_time               float64
active_hour_여행/교통_usage_time            float64
active_hour_건강_usage_time               float64
active_hour_음악_usage_time               float64
active_hour_사진/영상_usage_time            float64
active_hour_소셜_usage_time               float64
active_hour_생산성_usage_time              float64
active_hour_유틸리티_usage_time             float64
active_hour_교육_usage_time               float64
active_hour_식음료_usage_time              float64
active_hour_뉴스/정보_usage_time            float64
active_hour_라이프스타일_usage_time           float64
sleep_hour_금융_usage_time                float64
sleep_hour_기타_usage_time                float64
sleep_hour_쇼핑_usage_time                float64
sleep_hour

,subject_id,lifelog_date,active_hour_금융_usage_time,active_hour_기타_usage_time,active_hour_쇼핑_usage_time,active_hour_게임_usage_time,active_hour_여행/교통_usage_time,active_hour_건강_usage_time,active_hour_음악_usage_time,active_hour_사진/영상_usage_time,active_hour_소셜_usage_time,active_hour_생산성_usage_time,active_hour_유틸리티_usage_time,active_hour_교육_usage_time,active_hour_식음료_usage_time,active_hour_뉴스/정보_usage_time,active_hour_라이프스타일_usage_time,sleep_hour_금융_usage_time,sleep_hour_기타_usage_time,sleep_hour_쇼핑_usage_time,sleep_hour_게임_usage_time,sleep_hour_여행/교통_usage_time,sleep_hour_건강_usage_time,sleep_hour_음악_usage_time,sleep_hour_사진/영상_usage_time,sleep_hour_소셜_usage_time,sleep_hour_생산성_usage_time,sleep_hour_유틸리티_usage_time,sleep_hour_교육_usage_time,sleep_hour_식음료_usage_time,sleep_hour_뉴스/정보_usage_time,sleep_hour_라이프스타일_usage_time
0,id01,2024-06-26,10.8237,38.3196,3.7006,0.0000,7.1007,42.0251,7.8183,0.1061,0.0000,0.0000,89.1021,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4221,0.0000,0.0000,0.0000,0.0000
1,id01,2024-06-27,89.5100,48.5645,57.5088,0.0000,0.2933,9.0272,9.5388,1.6092,0.0000,0.0000,220.4942,11.6824,0.0000,7.8528,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,id01,2024-06-28,38.5043,35.4421,26.1193,0.0000,0.3222,8.7883,8.5665,23.9184,0.0000,0.0000,177.0139,0.0000,0.0000,23.4591,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


# nan_stats:
                               missing_count  missing_ratio(%)
subject_id                                 0            0.0000
lifelog_date                               0            0.0000
active_hour_금융_usage_time                  0            0.0000
active_hour_기타_usage_time                  0            0.0000
active_hour_쇼핑_usage_time                  0            0.0000
active_hour_게임_usage_time                  0            0.0000
active_hour_여행/교통_usage_time               0            0.0000
active_hour_건강_usage_time                  0            0.0000
active_hour_음악_usage_time                  0            0.0000
active_hour_사진/영상_usage_time               0            0.0000
active_hour_소셜_usage_time                  0            0.0000
active_hour_생산성_usage_time                 0            0.0000
active_hour_유틸리티_usage_time                0            0.0000
active_hour_교육_usage_time                  0            0.0000
active_hour_식음료_usage_time                

### ✔️ mWifi 주변wifi 정보
- Wifi devices around individual subject.
 - -30 ~ -50 dBm	매우 강한 신호 (최적)
 - -51 ~ -60 dBm	강한 신호 (문제 없음)
 - -61 ~ -70 dBm	괜찮은 신호 (약간 느릴 수 있음)
 - -71 ~ -80 dBm	약한 신호 (끊김 주의)
 - -81 dBm 이하	매우 약한 신호 (거의 끊김)

In [22]:
def process_mWifi(df, threshold=-60):
    wifi = df["m_wifi"].values  # [ [{'bssid': 'a0:0f:37:9a:5d:8b', 'rssi': -78}, ...], ...]

    def _process_feature(wifi):
        if len(wifi) == 0:
            return 0, 0., 0.,

        bssids = set()
        rssis = []
        for wifi_data in wifi:
            for data in wifi_data:
                if data["rssi"] >= threshold:
                    bssids.add(data["bssid"])
                    rssis.append(data["rssi"])

        bssid_count = len(bssids)
        mean_rssi = np.mean(rssis) if len(rssis) > 0 else 0
        max_rssi = np.max(rssis) if len(rssis) > 0 else 0

        return bssid_count, mean_rssi, max_rssi
    
    # 하루
    active_hour_bssid_count, active_hour_mean_rssi, active_hour_max_rssi = _process_feature(wifi[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_bssid_count, sleep_hour_mean_rssi, sleep_hour_max_rssi = _process_feature(wifi[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_bssid_count': active_hour_bssid_count,
        'active_hour_mean_rssi': active_hour_mean_rssi,
        'active_hour_max_rssi': active_hour_max_rssi,
        'sleep_hour_bssid_count': sleep_hour_bssid_count,
        'sleep_hour_mean_rssi': sleep_hour_mean_rssi,
        'sleep_hour_max_rssi': sleep_hour_max_rssi
    })

mWifi_ori = load_data(DataType.mWifi)
mWifi_ori = shift_lifelog_date(mWifi_ori, target_hours=SLEEP_HOURS)

mWifi2 = (
    mWifi_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mWifi)
    .reset_index(drop=True)
)
describe_df(mWifi2)

# shape:
(784, 8)

# dtypes:
subject_id                       category
lifelog_date               datetime64[ns]
active_hour_bssid_count           float64
active_hour_mean_rssi             float64
active_hour_max_rssi              float64
sleep_hour_bssid_count            float64
sleep_hour_mean_rssi              float64
sleep_hour_max_rssi               float64
dtype: object



,subject_id,lifelog_date,active_hour_bssid_count,active_hour_mean_rssi,active_hour_max_rssi,sleep_hour_bssid_count,sleep_hour_mean_rssi,sleep_hour_max_rssi
0,id01,2024-06-26,50.0000,-50.2712,-19.0000,6.0000,-40.0566,-27.0000
1,id01,2024-06-27,12.0000,-47.0130,-26.0000,4.0000,-40.7407,-27.0000
2,id01,2024-06-28,21.0000,-47.2000,-26.0000,6.0000,-38.5932,-28.0000


# nan_stats:
                         missing_count  missing_ratio(%)
subject_id                           0            0.0000
lifelog_date                         0            0.0000
active_hour_bssid_count              0            0.0000
active_hour_mean_rssi                0            0.0000
active_hour_max_rssi                 0            0.0000
sleep_hour_bssid_count               0            0.0000
sleep_hour_mean_rssi                 0            0.0000
sleep_hour_max_rssi                  0            0.0000



### ✔️ wHr 심박동수
- Heart rate readings recorded by the smartwatch.

In [23]:
def process_wHr(df):
    heart_rate = df["heart_rate"].values  # [[0, 1, 2, ...], ...]

    def _process_feature(heart_rate):
        if len(heart_rate) == 0:
            return 0., 0., 0., 0., 0.

        heart_rate = np.array(sum(map(lambda x: x.tolist(), heart_rate), []))
        mean_hr = heart_rate.mean() if len(heart_rate) > 0 else 0
        min_hr = heart_rate.min() if len(heart_rate) > 0 else 0
        max_hr = heart_rate.max() if len(heart_rate) > 0 else 0
        std_hr = heart_rate.std() if len(heart_rate) > 0 else 0
        high_hr = heart_rate[heart_rate > 100].sum()

        return mean_hr, min_hr, max_hr, std_hr, high_hr

    # 하루
    active_hour_mean_hr, active_hour_min_hr, active_hour_max_hr, active_hour_std_hr, active_hour_high_hr = _process_feature(heart_rate[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_mean_hr, sleep_hour_min_hr, sleep_hour_max_hr, sleep_hour_std_hr, sleep_hour_high_hr = _process_feature(heart_rate[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_mean_hr': active_hour_mean_hr,
        'active_hour_min_hr': active_hour_min_hr,
        'active_hour_max_hr': active_hour_max_hr,
        'active_hour_std_hr': active_hour_std_hr,
        'active_hour_high_hr': active_hour_high_hr,
        'sleep_hour_mean_hr': sleep_hour_mean_hr,
        'sleep_hour_min_hr': sleep_hour_min_hr,
        'sleep_hour_max_hr': sleep_hour_max_hr,
        'sleep_hour_std_hr': sleep_hour_std_hr,
        'sleep_hour_high_hr': sleep_hour_high_hr
    })

wHr_ori = load_data(DataType.wHr)
wHr_ori = shift_lifelog_date(wHr_ori, target_hours=SLEEP_HOURS)

wHr2 = (
    wHr_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_wHr)
    .reset_index(drop=True)
)

describe_df(wHr2)

# shape:
(679, 12)

# dtypes:
subject_id                   category
lifelog_date           datetime64[ns]
active_hour_mean_hr           float64
active_hour_min_hr            float64
active_hour_max_hr            float64
active_hour_std_hr            float64
active_hour_high_hr           float64
sleep_hour_mean_hr            float64
sleep_hour_min_hr             float64
sleep_hour_max_hr             float64
sleep_hour_std_hr             float64
sleep_hour_high_hr            float64
dtype: object



,subject_id,lifelog_date,active_hour_mean_hr,active_hour_min_hr,active_hour_max_hr,active_hour_std_hr,active_hour_high_hr,sleep_hour_mean_hr,sleep_hour_min_hr,sleep_hour_max_hr,sleep_hour_std_hr,sleep_hour_high_hr
0,id01,2024-06-26,81.2434,59.0000,142.0000,11.8712,243191.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,id01,2024-06-27,79.3523,53.0000,130.0000,12.6371,119052.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,id01,2024-06-28,77.3601,51.0000,135.0000,12.5109,142587.0000,0.0000,0.0000,0.0000,0.0000,0.0000


# nan_stats:
                     missing_count  missing_ratio(%)
subject_id                       0            0.0000
lifelog_date                     0            0.0000
active_hour_mean_hr              0            0.0000
active_hour_min_hr               0            0.0000
active_hour_max_hr               0            0.0000
active_hour_std_hr               0            0.0000
active_hour_high_hr              0            0.0000
sleep_hour_mean_hr               0            0.0000
sleep_hour_min_hr                0            0.0000
sleep_hour_max_hr                0            0.0000
sleep_hour_std_hr                0            0.0000
sleep_hour_high_hr               0            0.0000



### ✔️ wLight 앰비언트 라이트
- Ambient light measured by the smartwatch.  
  - 어두운 밤 0.1 ~ 1 lux 캄캄한 방, 달빛 없는 밤
  - 가로등 켜진 거리 10 ~ 20 lux 흐릿한 외부 조명
  - 실내 조명 100 ~ 500 lux 사무실, 일반 거실
  - 밝은 실외 10,000 ~ 25,000 lux 맑은 날 햇빛
  - 직사광선 아래 30,000 ~ 100,000 lux 여름 한낮, 매우 강한 햇빛

In [24]:
# mLight 와 같은 함수 사용!!!
wLight_ori = load_data(DataType.wLight)
wLight_ori = shift_lifelog_date(wLight_ori, target_hours=SLEEP_HOURS)

wLight2_ = (
    wLight_ori
    .rename(columns={"w_light": "m_light"})
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mLight)
    .reset_index(drop=True)
)

wLight2_.rename(
    columns={
        col: "wlight_" + col.replace("mlight_", "")
        for col in wLight2_.columns if col not in ["subject_id", "lifelog_date"]
    }, inplace=True
)
wLight2 = wLight2_.copy()
describe_df(wLight2)

# shape:
(752, 14)

# dtypes:
subject_id                             category
lifelog_date                     datetime64[ns]
wlight_active_hour_mean_light           float64
wlight_active_hour_min_light            float64
wlight_active_hour_max_light            float64
wlight_active_hour_std_light            float64
wlight_sleep_hour_mean_light            float64
wlight_sleep_hour_min_light             float64
wlight_sleep_hour_max_light             float64
wlight_sleep_hour_std_light             float64
wlight_first_sleep_minutes              float64
wlight_first_wakeup_minutes             float64
wlight_sleep_duration                   float64
wlight_sleep_transitions                float64
dtype: object



,subject_id,lifelog_date,wlight_active_hour_mean_light,wlight_active_hour_min_light,wlight_active_hour_max_light,wlight_active_hour_std_light,wlight_sleep_hour_mean_light,wlight_sleep_hour_min_light,wlight_sleep_hour_max_light,wlight_sleep_hour_std_light,wlight_first_sleep_minutes,wlight_first_wakeup_minutes,wlight_sleep_duration,wlight_sleep_transitions
0,id01,2024-06-26,299.4151,0.0000,20874.0000,1220.1126,0.0000,0.0000,0.0000,0.0000,1274.0000,1980.0000,706.0000,0.0000
1,id01,2024-06-27,290.7522,0.0000,12464.0000,1031.2252,0.0000,0.0000,0.0000,0.0000,1200.0000,1830.0000,630.0000,0.0000
2,id01,2024-06-28,518.8263,0.0000,91584.0000,3794.5034,0.0000,0.0000,0.0000,0.0000,1209.0000,1835.0000,626.0000,0.0000


# nan_stats:
                               missing_count  missing_ratio(%)
subject_id                                 0            0.0000
lifelog_date                               0            0.0000
wlight_active_hour_mean_light              0            0.0000
wlight_active_hour_min_light               0            0.0000
wlight_active_hour_max_light               0            0.0000
wlight_active_hour_std_light               0            0.0000
wlight_sleep_hour_mean_light               0            0.0000
wlight_sleep_hour_min_light                0            0.0000
wlight_sleep_hour_max_light                0            0.0000
wlight_sleep_hour_std_light                0            0.0000
wlight_first_sleep_minutes                 0            0.0000
wlight_first_wakeup_minutes                0            0.0000
wlight_sleep_duration                      0            0.0000
wlight_sleep_transitions                   0            0.0000



In [25]:
wLight2.wlight_sleep_duration.describe()

count   752.0000
mean    641.3045
std     118.6218
min     274.0000
25%     556.7500
50%     652.0000
75%     723.0000
max     839.0000
Name: wlight_sleep_duration, dtype: float64

### ✔️ wPedo 걸음수
- Step data recorded by the smartwatch.

In [26]:
def process_mPedo(df):

    def _process_feature(df):
        if len(df) == 0:
            return 0., 0., 0.

        steps = df["step"].values
        distances = df["distance"].values
        calories = df["burned_calories"].values

        steps = steps.sum() if len(steps) > 0 else 0
        distance = distances.sum() if len(distances) > 0 else 0
        burned_calories = calories.sum() if len(calories) > 0 else 0

        return steps, distance, burned_calories

    # 하루
    active_hour_steps, active_hour_distance, active_hour_burned_calories = _process_feature(df[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_steps, sleep_hour_distance, sleep_hour_burned_calories = _process_feature(df[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_steps': active_hour_steps,
        'active_hour_distance': active_hour_distance,
        'active_hour_burned_calories': active_hour_burned_calories,
        'sleep_hour_steps': sleep_hour_steps,
        'sleep_hour_distance': sleep_hour_distance,
        'sleep_hour_burned_calories': sleep_hour_burned_calories
    })

wPedo_ori = load_data(DataType.wPedo)
wPedo_ori = shift_lifelog_date(wPedo_ori, target_hours=SLEEP_HOURS)

wPedo2 = (
    wPedo_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mPedo)
    .reset_index(drop=True)
)

describe_df(wPedo2)

# shape:
(735, 8)

# dtypes:
subject_id                           category
lifelog_date                   datetime64[ns]
active_hour_steps                     float64
active_hour_distance                  float64
active_hour_burned_calories           float64
sleep_hour_steps                      float64
sleep_hour_distance                   float64
sleep_hour_burned_calories            float64
dtype: object



,subject_id,lifelog_date,active_hour_steps,active_hour_distance,active_hour_burned_calories,sleep_hour_steps,sleep_hour_distance,sleep_hour_burned_calories
0,id01,2024-06-26,3578.0000,2782.1901,189.3191,0.0000,0.0000,0.0000
1,id01,2024-06-27,2619.0000,2020.5527,280.2708,10.0000,6.8300,0.0000
2,id01,2024-06-28,3726.0000,2888.0892,116.1595,0.0000,0.0000,0.0000


# nan_stats:
                             missing_count  missing_ratio(%)
subject_id                               0            0.0000
lifelog_date                             0            0.0000
active_hour_steps                        0            0.0000
active_hour_distance                     0            0.0000
active_hour_burned_calories              0            0.0000
sleep_hour_steps                         0            0.0000
sleep_hour_distance                      0            0.0000
sleep_hour_burned_calories               0            0.0000



### 📦 merge 데이터

In [27]:
df_list = [
    mACStatus2,       # 1
    mActivity2,       # 2
    mAmbience2,       # 3
    mBle2,            # 4
    mGps2,            # 5
    mLight2,          # 6
    mScreenStatus2,   # 7
    mUsageStats2,     # 8
    mWifi2,           # 9
    wHr2,             # 10
    wLight2,          # 11
    wPedo2,           # 12
]

data = reduce(lambda left, right: pd.merge(left, right, on=['subject_id', 'lifelog_date'], how='outer'), df_list)

# 중복체크
print(data.shape)
print(data[['subject_id','lifelog_date']].drop_duplicates().shape)


(806, 140)
(806, 2)


### 🤞 공통 후처리

#### feature 드랍

In [28]:
drop_features = ['top_bssid'] # ,'week_type','week_type_lag1'
drop_features = [i for i in drop_features if i in data.columns.tolist()]
data = data.drop(columns=drop_features, errors='ignore')

#### 요일, 공휴일, 추정 휴가

In [29]:
weekday_map = {
    0: '월요일', 1: '화요일', 2: '수요일', 3: '목요일',
    4: '금요일', 5: '토요일', 6: '일요일'
}

data["weekday"] = data["lifelog_date"].dt.dayofweek.map(weekday_map)
data["month"] = data["lifelog_date"].dt.month
data["weekend"] = data["weekday"].isin(['토요일', '일요일']).astype(int)
data["holiday"] = data["lifelog_date"].isin(HOLIDAY_DATES).astype(int)
data["weekend_holiday"] = (data["weekend"] | data["holiday"]).astype(int)

In [30]:
# 평균보다 한시간 많이 잔 날을 추정 휴가로 간주
def apply_rule_based_holiday(df):
    mean_sleep_duration = df['mlight_sleep_duration'].mean()
    df['rule_based_holiday'] = (df['mlight_sleep_duration'] > mean_sleep_duration + 60).astype(int)
    return df

data = (
    data
    .groupby('subject_id', group_keys=False, as_index=False, sort=False, observed=True)
    .apply(apply_rule_based_holiday)
)

In [31]:
data.head(3)

,subject_id,lifelog_date,charging_ratio,charging_sum,charging_transitions,avg_charging_duration,max_charging_duration,sleep_charging_ratio,sleep_charging_sum,sleep_charging_transitions,sleep_avg_charging_duration,sleep_max_charging_duration,walking_minutes,vehicle_minutes,activity_minutes,sleep_walking_minutes,sleep_vehicle_minutes,sleep_activity_minutes,met_mean,met_sum,active_hour_unique_label_count,active_hour_snor_count,sleep_hour_unique_label_count,sleep_hour_snor_count,work_hour_rssi_mean,work_hour_rssi_min,work_hour_rssi_max,work_hour_others_ratio,work_hour_unknown_ratio,free_hour_rssi_mean,free_hour_rssi_min,free_hour_rssi_max,free_hour_others_ratio,free_hour_unknown_ratio,sleep_hour_rssi_mean,sleep_hour_rssi_min,sleep_hour_rssi_max,sleep_hour_others_ratio,sleep_hour_unknown_ratio,active_hour_walk_minutes,active_hour_jog_minutes,active_hour_vehicle_minutes,active_hour_mean_speed,active_hour_max_speed,active_hour_min_speed,active_hour_distance_x,exercise_flag,sleep_hour_walk_minutes,sleep_hour_jog_minutes,sleep_hour_vehicle_minutes,sleep_hour_mean_speed,sleep_hour_max_speed,sleep_hour_min_speed,sleep_hour_distance_x,mgps_first_wakeup_minutes,active_hour_mean_light,active_hour_min_light,active_hour_max_light,active_hour_std_light,sleep_hour_mean_light,sleep_hour_min_light,sleep_hour_max_light,sleep_hour_std_light,mlight_first_sleep_minutes,mlight_first_wakeup_minutes,mlight_sleep_duration,mlight_sleep_transitions,screen_use_ratio,screen_use_sum,screen_use_transitions,sleep_screen_use_ratio,sleep_screen_use_sum,sleep_screen_use_transitions,mscreen_first_sleep_minutes,mscreen_first_wakeup_minutes,mscreen_sleep_duration,active_hour_금융_usage_time,active_hour_기타_usage_time,active_hour_쇼핑_usage_time,active_hour_게임_usage_time,active_hour_여행/교통_usage_time,active_hour_건강_usage_time,active_hour_음악_usage_time,active_hour_사진/영상_usage_time,active_hour_소셜_usage_time,active_hour_생산성_usage_time,active_hour_유틸리티_usage_time,active_hour_교육_usage_time,active_hour_식음료_usage_time,active_hour_뉴스/정보_usage_time,active_hour_라이프스타일_usage_time,sleep_hour_금융_usage_time,sleep_hour_기타_usage_time,sleep_hour_쇼핑_usage_time,sleep_hour_게임_usage_time,sleep_hour_여행/교통_usage_time,sleep_hour_건강_usage_time,sleep_hour_음악_usage_time,sleep_hour_사진/영상_usage_time,sleep_hour_소셜_usage_time,sleep_hour_생산성_usage_time,sleep_hour_유틸리티_usage_time,sleep_hour_교육_usage_time,sleep_hour_식음료_usage_time,sleep_hour_뉴스/정보_usage_time,sleep_hour_라이프스타일_usage_time,active_hour_bssid_count,active_hour_mean_rssi,active_hour_max_rssi,sleep_hour_bssid_count,sleep_hour_mean_rssi,sleep_hour_max_rssi,active_hour_mean_hr,active_hour_min_hr,active_hour_max_hr,active_hour_std_hr,active_hour_high_hr,sleep_hour_mean_hr,sleep_hour_min_hr,sleep_hour_max_hr,sleep_hour_std_hr,sleep_hour_high_hr,wlight_active_hour_mean_light,wlight_active_hour_min_light,wlight_active_hour_max_light,wlight_active_hour_std_light,wlight_sleep_hour_mean_light,wlight_sleep_hour_min_light,wlight_sleep_hour_max_light,wlight_sleep_hour_std_light,wlight_first_sleep_minutes,wlight_first_wakeup_minutes,wlight_sleep_duration,wlight_sleep_transitions,active_hour_steps,active_hour_distance_y,active_hour_burned_calories,sleep_hour_steps,sleep_hour_distance_y,sleep_hour_burned_calories,weekday,month,weekend,holiday,weekend_holiday,rule_based_holiday
0,id01,2024-06-26,0.1498,147.0000,22.0000,13.3636,41.0000,0.0000,0.0000,0.0000,0.0000,0.0000,32.0000,89.0000,121.0000,0.0000,0.0000,0.0000,2.0196,2041.8000,265.0000,2.0000,10.0000,0.0000,-74.0904,-94.0000,-27.0000,0.0590,0.9410,-77.2213,-92.0000,-43.0000,0.0791,0.9209,0.0000,0.0000,0.0000,0.0000,0.0000,68.0000,32.0000,19.0000,0.5775,19.0505,0.0000,16.7900,1.0000,36.0000,0.0000,0.0000,0.1812,1.6664,0.0000,0.1752,1980.0000,364.5068,0.0000,1886.0000,392.9401,0.0000,0.0000,0.0000,0.0000,1203.0000,1980.0000,777.0000,0.0000,0.2098,210.0000,54.0000,0.0000,0.0000,0.0000,1440.0000,1980.0000,540.0000,10.8237,38.3196,3.7006,0.0000,7.1007,42.0251,7.8183,0.1061,0.0000,0.0000,89.1021,0.0000,0.0000,0.000

#### ✔️ 결측치 처리

In [32]:
num_columns = data.select_dtypes(include=[np.number]).columns
data[num_columns] = data[num_columns].fillna(-1)

### 전처리 완료 데이터 저장

In [33]:
train = load_train()
test = load_test()

In [34]:

# merge
train2 = train.merge(data, on=['subject_id','lifelog_date'], how='left')
test2 = test.merge(data, on=['subject_id','lifelog_date'], how='left')

# 저장
print('# train  shape:',train.shape)
print('# train2 shape:',test2.shape)
print('# test   shape:',test.shape)
print('# test2  shape:',test2.shape)

# train  shape: (450, 9)
# train2 shape: (250, 153)
# test   shape: (250, 9)
# test2  shape: (250, 153)


In [36]:
# 저장
DATA_VERSION = f"{datetime.now().strftime('%Y%m%d')}"
train2.to_parquet(DATA_DIR / f"train_{DATA_VERSION}.parquet")
test2.to_parquet(DATA_DIR / f"test_{DATA_VERSION}.parquet")

### 📌 모델 학습

In [37]:
train = pd.read_parquet(DATA_DIR / f"train_{DATA_VERSION}.parquet")
test = pd.read_parquet(DATA_DIR / f"test_{DATA_VERSION}.parquet")

In [145]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils.class_weight import compute_sample_weight

In [146]:
lgb_A = 0.3
xgb_B = 0.3
cat_C = 0.4

In [147]:
def get_oof_predictions(X, y, lgb_params, xgb_params, n_splits=5, is_multiclass=False, num_class=None, early_stop=False):
    oof_preds_lgb = np.zeros(len(X))
    oof_preds_xgb = np.zeros(len(X))
    oof_preds_cat = np.zeros(len(X))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for train_idx, valid_idx in skf.split(X, y):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        # LightGBM
        if is_multiclass:
            lgb_model = LGBMClassifier(**lgb_params, objective='multiclass', num_class=num_class)
        else:
            lgb_model = LGBMClassifier(**lgb_params)

        # XGBoost
        if is_multiclass:
            xgb_model = XGBClassifier(**xgb_params, objective='multi:softmax', num_class=num_class)
        else:
            xgb_model = XGBClassifier(**xgb_params)

        # CatBoost
        if is_multiclass:
            cat_model = CatBoostClassifier(**common_params_cat2, objective='MultiClass', classes_count=num_class)
        else:
            cat_model = CatBoostClassifier(**common_params_cat)

        if early_stop:
            lgb_model.fit(
                X_train, y_train,
                eval_set=[(X_train, y_train), (X_valid, y_valid)],
                callbacks=[early_stopping(stopping_rounds=100, verbose=False)]
            )
            xgb_model.fit(
                X_train, y_train,
                eval_set=[(X_valid, y_valid)],
                early_stopping_rounds=100,
                verbose=False
            )
            cat_model.fit(
                X_train, y_train,
                eval_set=[(X_valid, y_valid)],
                early_stopping_rounds=100,
                verbose=False
            )
        else:
            if is_multiclass:

                # 클래스 weight 계산
                classes = np.unique(y_train)
                weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
                class_weights = dict(zip(classes, weights))
                
                # 각 샘플에 대해 weight 매핑
                w_train = pd.Series(y_train).map(class_weights)
                #print(w_train)

                w_train = compute_sample_weight(class_weight='balanced', y=y_train)
                # print(w_train)
                
                lgb_model.fit(X_train, y_train, sample_weight=w_train)
                xgb_model.fit(X_train, y_train, sample_weight=w_train)
                cat_model.fit(X_train, y_train)
            else:
                lgb_model.fit(X_train, y_train)
                xgb_model.fit(X_train, y_train)
                cat_model.fit(X_train, y_train)
            

        # Get predictions
        lgb_preds = lgb_model.predict(X_valid)
        xgb_preds = xgb_model.predict(X_valid)
        cat_preds = cat_model.predict(X_valid).ravel()  # ✅ 2차원 → 1차원
        
        # Store predictions
        oof_preds_lgb[valid_idx] = lgb_preds
        oof_preds_xgb[valid_idx] = xgb_preds
        oof_preds_cat[valid_idx] = cat_preds

    # Ensemble predictions (7:3 ratio)
    oof_preds = lgb_A * oof_preds_lgb + xgb_B * oof_preds_xgb + cat_C * oof_preds_cat
    
    if not is_multiclass:
        oof_preds = (oof_preds > 0.5).astype(int)
    else:
        oof_preds = np.round(oof_preds).astype(int)

    return oof_preds

In [153]:
def run_basemodel(train, test, valid_ids, common_params, n_splits, random_state=42, early_stop=False):
    #Best
    lgb_A = 0.3
    xgb_B = 0.3
    cat_C = 0.4
    
    train_df = train.copy()
    test_df = test.copy()

    submission_final = test_df[['subject_id', 'sleep_date', 'lifelog_date']].copy()
    submission_final['lifelog_date'] = pd.to_datetime(submission_final['lifelog_date']).dt.date

    # 타겟
    targets_binary = ['Q1', 'Q2', 'Q3', 'S2', 'S3']
    targets_binary_name = ['기상직후수면질','취침전신체적피로','취침전스트레스','수면효율','수면잠들기시간']
    target_multiclass = 'S1'
    all_targets = targets_binary + [target_multiclass]

    # 노이즈 수준 설정
    def add_noise(series, noise_level, seed=3):
        rng = np.random.default_rng(seed)
        return series * (1 + noise_level * rng.standard_normal(len(series)))

    noise_level = 0.015  # 필요에 따라 조정

    # 타겟인코딩
    # m = 0: 스무딩 없이 범주별 평균만 사용합니다. 관측 수가 많은 범주에는 적합하지만, 적은 경우 과적합 위험이 있습니다.
    # m = 1~10: 일반적인 기본값으로, 대부분의 상황에서 안정적인 성능을 보입니다.
    # m = 50~300: 관측 수가 매우 적은 범주가 많거나 데이터가 희소한 경우에 유용합니다.
    for tgt in all_targets:

      encoder_feats = ['subject_id','month','weekend'] # 'weekday', 'subject_id','month','weekend'

      #### 타겟인코딩1

      subject_mean = train_df.groupby(encoder_feats)[tgt].mean().rename(f'{tgt}_te')
      train_df = train_df.merge(subject_mean, on=encoder_feats, how='left')
      test_df = test_df.merge(subject_mean, on=encoder_feats, how='left')
      global_mean = train_df[tgt].mean()
      test_df[f'{tgt}_te'] = test_df[f'{tgt}_te'].fillna(global_mean)

      # 노이즈 추가
      train_df[f'{tgt}_te'] = add_noise(train_df[f'{tgt}_te'], noise_level)
      test_df[f'{tgt}_te'] = add_noise(test_df[f'{tgt}_te'], noise_level)

      #### 타겟인코딩2

      # 새로운 범주형 열 생성
      train_df['TMP'] = train_df[encoder_feats].applymap(str).apply(lambda x: ''.join(x) ,axis=1)
      test_df['TMP'] = test_df[encoder_feats].applymap(str).apply(lambda x: ''.join(x) ,axis=1)

      # 인코더
      encoder = TargetEncoder(cols=['TMP'], smoothing=300) # 40
      encoder.fit(train_df[['TMP']], train_df[tgt])

      # 인코딩 결과를 새로운 열에 저장
      train_df[f'{tgt}_te2'] = encoder.transform(train_df[['TMP']])
      test_df[f'{tgt}_te2'] = encoder.transform(test_df[['TMP']])

      # 노이즈 추가
      train_df[f'{tgt}_te2'] = add_noise(train_df[f'{tgt}_te2'], noise_level)
      test_df[f'{tgt}_te2'] = add_noise(test_df[f'{tgt}_te2'], noise_level)

      # 불필요한 변수 제거
      train_df = train_df.drop(columns=['TMP'])
      test_df = test_df.drop(columns=['TMP'])


    # 인코딩
    PK = ['sleep_date', 'lifelog_date', 'subject_id']
    encoder = LabelEncoder()
    categorical_features = [i for i in train_df.select_dtypes(include=['object', 'category']).columns if i not in PK+['pk']]
    for col in categorical_features:
        print(col)
        train_df[col] = encoder.fit_transform(train_df[col])
        test_df[col] = encoder.fit_transform(test_df[col])


    # X
    X = train_df.drop(columns=PK + all_targets)
    test_X = test_df.drop(columns=PK + all_targets)
    print(f'# X shape: {X.shape}')
    print(f'# test_X shape: {test_X.shape}')

    print('\n STEP1: 실험 결과 확인')
    print("=============== Validation Results ==============")
    total_avg_f1s = []
    best_iteration_temp = {k: [] for k in all_targets}

    val_f1 = []
    
    binary_val_preds = {}
    multiclass_val_preds = {}

    binary_test_preds = {}
    multiclass_test_preds = {}
    test_preds = {}

    # Find optimal weights
    best_weights = []
    best_scores = []
    
    for col in targets_binary:
        # binary
        y = train_df[col]

        valid_ids['pk'] = valid_ids['subject_id']+valid_ids['sleep_date']
        train_df['pk'] = train_df['subject_id']+train_df['sleep_date']

        X_valid = train_df.loc[train_df['pk'].isin(valid_ids['pk']),X.columns.tolist()].reset_index(drop=True).copy()
        X_train = train_df.loc[~train_df['pk'].isin(valid_ids['pk']),X.columns.tolist()].reset_index(drop=True).copy()
        y_valid = train_df.loc[train_df['pk'].isin(valid_ids['pk']),y.name].reset_index(drop=True).copy()
        y_train = train_df.loc[~train_df['pk'].isin(valid_ids['pk']),y.name].reset_index(drop=True).copy()

        # Get parameters for both models
        lgb_params = common_params[col].copy()
        lgb_params['random_state'] = random_state
        
        xgb_params = {
            'n_estimators': 1000,
            'learning_rate': 0.01,
            'max_depth': 6,
            'min_child_weight': 1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': random_state
        }

        # Train LightGBM
        lgb_model = LGBMClassifier(**lgb_params)
        if early_stop:
            lgb_model.fit(
                X_train, y_train,
                eval_set=[(X_train, y_train), (X_valid, y_valid)],
                callbacks=[early_stopping(stopping_rounds=100,verbose=False)]
            )
            best_iteration_temp[col].append(lgb_model.best_iteration_)
        else:
            lgb_model.fit(X_train, y_train)
            best_iteration_temp[col].append(1000)

        # Train XGBoost
        xgb_model = XGBClassifier(**xgb_params)
        if early_stop:
            xgb_model.fit(
                X_train, y_train,
                eval_set=[(X_valid, y_valid)],
                early_stopping_rounds=100,
                verbose=False
            )
        else:
            xgb_model.fit(X_train, y_train)

        # Train Catboost
        cat_model = CatBoostClassifier(**common_params_cat, loss_function='Logloss')
        if early_stop:
            cat_model.fit(
                X_train, y_train,
                eval_set=[(X_valid, y_valid)],
                early_stopping_rounds=100,
                verbose=False
            )
        else:
            cat_model.fit(X_train, y_train)

        # Get predictions and ensemble
        lgb_pred_valid = lgb_model.predict_proba(X_valid)[:, 1]
        xgb_pred_valid = xgb_model.predict_proba(X_valid)[:, 1]
        cat_pred_valid = cat_model.predict_proba(X_valid)[:, 1]
        pred_valid = (lgb_A * lgb_pred_valid + xgb_B * xgb_pred_valid + cat_C * cat_pred_valid  > 0.5).astype(int)
        
        f1 = f1_score(y_valid, pred_valid, average='macro')
        val_f1.append(f1)

        # Store predictions
        binary_val_preds[col] = {
            'lgb': lgb_pred_valid,
            'xgb': xgb_pred_valid,
            'cat': cat_pred_valid,
            'true': y_valid
        }

    # multiclass
    y = train_df[target_multiclass]

    X_valid = train_df.loc[train_df['pk'].isin(valid_ids['pk']),X.columns.tolist()].reset_index(drop=True).copy()
    X_train = train_df.loc[~train_df['pk'].isin(valid_ids['pk']),X.columns.tolist()].reset_index(drop=True).copy()
    y_valid = train_df.loc[train_df['pk'].isin(valid_ids['pk']),y.name].reset_index(drop=True).copy()
    y_train = train_df.loc[~train_df['pk'].isin(valid_ids['pk']),y.name].reset_index(drop=True).copy()

    # Get parameters for both models
    lgb_params = common_params['S1'].copy()
    lgb_params['random_state'] = random_state
    
    xgb_params = {
        'n_estimators': 1000,
        'learning_rate': 0.01,
        'max_depth': 6,
        'min_child_weight': 1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': random_state
    }

    # 클래스 weight 계산
    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weights = dict(zip(classes, weights))
    
    # 각 샘플에 대해 weight 매핑
    w_train = pd.Series(y_train).map(class_weights)
    # print("----compute_class_weight:")
    # print(w_train)

    w_train = compute_sample_weight(class_weight='balanced', y=y_train)
    # print("----compute_sample_weight:")
    # print(w_train)

    # Train LightGBM
    lgb_model = LGBMClassifier(**lgb_params, objective='multiclass', num_class=3)
    if early_stop:
        lgb_model.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_valid, y_valid)],
            callbacks=[early_stopping(stopping_rounds=100,verbose=False)], sample_weight=w_train
        )
        best_iteration_temp[target_multiclass].append(lgb_model.best_iteration_)
    else:
        lgb_model.fit(X_train, y_train, sample_weight=w_train)
        best_iteration_temp[target_multiclass].append(1000)

    # Train XGBoost
    xgb_model = XGBClassifier(**xgb_params, objective='multi:softmax', num_class=3)
    if early_stop:
        xgb_model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            early_stopping_rounds=100,
            verbose=False, sample_weight=w_train
        )
    else:
        xgb_model.fit(X_train, y_train,sample_weight=w_train)

    # Train Catboost
    cat_model = CatBoostClassifier(**common_params_cat2, loss_function='MultiClass', classes_count=3)
    if early_stop:
        cat_model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            early_stopping_rounds=100,
            verbose=False
        )
    else:
        cat_model.fit(X_train, y_train)

    # Get predictions and ensemble
    lgb_pred_valid = lgb_model.predict_proba(X_valid)
    xgb_pred_valid = xgb_model.predict_proba(X_valid)
    cat_pred_valid = cat_model.predict_proba(X_valid)
    pred_valid = np.argmax(lgb_A * lgb_pred_valid + xgb_B * xgb_pred_valid + cat_C * cat_pred_valid, axis=1)
    
    f1 = f1_score(y_valid, pred_valid, average='macro')
    val_f1.append(f1)

    multiclass_val_preds = {
        'lgb': lgb_pred_valid,
        'xgb': xgb_pred_valid,
        'cat': cat_pred_valid,
        'true': y_valid
    }

    # Generate all possible weight combinations that sum to 1
    step = 0.1
    for lgb_A in np.arange(0, 1.1, step):
        for xgb_B in np.arange(0, 1.1 - lgb_A, step):
            cat_C = 1.0 - lgb_A - xgb_B
            if cat_C < 0 or cat_C > 1:
                continue
            weights = (lgb_A, xgb_B, cat_C)
            print("========================================")
            print(f"\nTrying weights: lgb_A={lgb_A:.1f}, xgb_B={xgb_B:.1f}, cat_C={cat_C:.1f}")
            val_scores = []

            # Binary targets
            for col in targets_binary:
                preds = binary_val_preds[col]
                ensemble_pred = (lgb_A * preds['lgb'] +
                                 xgb_B * preds['xgb'] +
                                 cat_C * preds['cat'] > 0.5).astype(int)

                f1 = f1_score(preds['true'], ensemble_pred, average='macro')

                val_scores.append(f1)

                print(f" Validation Score {col}: {f1:.4f}")

            # Multiclass target
            preds = multiclass_val_preds
            ensemble_pred = np.argmax(lgb_A * preds['lgb'] + xgb_B * preds['xgb'] + cat_C * preds['cat'], axis=1)
            f1 = f1_score(preds['true'], ensemble_pred, average='macro')

            print(f" Validation Score S1: {f1:.4f}")
            val_scores.append(f1)

            avg_score = np.mean(val_scores)
            best_weights.append(weights)
            best_scores.append(avg_score)
            print(f"Average Validation Score: {avg_score:.4f}")

    # Sort results and get all
    sorted_indices = np.argsort(best_scores)[::-1]
    top_3_weights = [best_weights[i] for i in sorted_indices]
    top_3_scores = [best_scores[i] for i in sorted_indices]
    
    print("\nTop All Weight Combinations:")
    for i, (weights, score) in enumerate(zip(top_3_weights, top_3_scores)):
        print(f"Rank {i+1}: lgb_A={weights[0]:.1f}, xgb_B={weights[1]:.1f}, cat_C={weights[2]:.1f} - Score: {score:.4f}")
        

    avg_f1 = np.mean(val_f1)
    total_avg_f1s.append(avg_f1)
    detail = " ".join([f"{name}({tname}):{score:.4f}" for name, tname, score in zip(targets_binary + [target_multiclass], targets_binary_name + ['S1'], val_f1)])
    print(f" 평균 F1: {avg_f1:.4f} / [상세] {detail}")

    best_iteration_dict = {k: max(best_iteration_temp[k]) for k in all_targets}

    if early_stop==True:
      print("\n[best_iteration_dict]")
      for k, v in best_iteration_dict.items():
          print(f"{k}: {v}")


    print(f"# 전체 평균 F1: {np.mean(total_avg_f1s):.4f}")
    print("================================================")

    # modoling with 100% train & no valid
    print('\n STEP2: 전체 데이터로 모델 재학습')
    print("====== modeling with 100% train & no valid =====")

    # binary
    binary_preds = {}
    binary_preds_proba = {}
    for col in targets_binary:
        # Get parameters for both models
        lgb_params = common_params[col].copy()
        lgb_params['random_state'] = random_state
        
        xgb_params = {
            'n_estimators': 1000,
            'learning_rate': 0.01,
            'max_depth': 6,
            'min_child_weight': 1,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': random_state
        }

        y = train_df[col]

        if early_stop:
            lgb_params['n_estimators'] = best_iteration_dict[col]
            xgb_params['n_estimators'] = best_iteration_dict[col]

        # Train LightGBM
        lgb_model = LGBMClassifier(**lgb_params)
        lgb_model.fit(X, y)

        # Train XGBoost
        xgb_model = XGBClassifier(**xgb_params)
        xgb_model.fit(X, y)

        # Train CatBoost
        cat_model = CatBoostClassifier(**common_params_cat)
        cat_model.fit(X, y)

        # Get predictions and ensemble
        lgb_pred = lgb_model.predict_proba(test_X)[:, 1]
        xgb_pred = xgb_model.predict_proba(test_X)[:, 1]
        cat_pred = cat_model.predict_proba(test_X)[:, 1]
        binary_preds[col] = (lgb_A * lgb_pred + xgb_B * xgb_pred + cat_C * cat_pred > 0.5).astype(int)
        binary_preds_proba[col] = lgb_A * lgb_model.predict_proba(test_X) + xgb_B * xgb_model.predict_proba(test_X) + cat_C * cat_model.predict_proba(test_X)

        # Store predictions
        binary_test_preds[col] = {
            'lgb': lgb_pred,
            'xgb': xgb_pred,
            'cat': cat_pred
        }
        
        # Feature importance (using LightGBM's importance)
        print("lightgbm feature importance")
        fi_df = pd.DataFrame({'feature': X.columns, 'importance': lgb_model.feature_importances_})
        top10 = fi_df.sort_values(by='importance', ascending=False).head(10)
        feat_str = ", ".join([f"{row['feature']}({int(row['importance'])})" for _, row in top10.iterrows()])
        print(f"[{col}] {feat_str}")

        # Feature importance (using XGBoost's importance)
        print("xgboost feature importance")
        fi_df = pd.DataFrame({
            'feature': X.columns,
            'importance': xgb_model.feature_importances_
        })
        top10 = fi_df.sort_values(by='importance', ascending=False).head(10)
        feat_str = ", ".join([f"{row['feature']}({int(row['importance'])})" for _, row in top10.iterrows()])
        print(f"[{col}] {feat_str}")

        # Feature importance (using CatBoost's importance)
        print("catboost feature importance")
        fi_df = pd.DataFrame({
            'feature': X.columns,
            'importance': cat_model.get_feature_importance()
        })
        top10 = fi_df.sort_values(by='importance', ascending=False).head(10)
        feat_str = ", ".join([f"{row['feature']}({int(row['importance'])})" for _, row in top10.iterrows()])
        print(f"[{col}] {feat_str}")

    # multiclass
    y = train_df['S1']
    
    # Get parameters for both models
    lgb_params = common_params['S1'].copy()
    lgb_params['random_state'] = random_state
    
    xgb_params = {
        'n_estimators': 1000,
        'learning_rate': 0.01,
        'max_depth': 6,
        'min_child_weight': 1,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': random_state
    }

    if early_stop:
        lgb_params['n_estimators'] = best_iteration_dict['S1']
        xgb_params['n_estimators'] = best_iteration_dict['S1']

    # 클래스 weight 계산
    classes = np.unique(y)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    class_weights = dict(zip(classes, weights))
    
    # 각 샘플에 대해 weight 매핑
    w_train = pd.Series(y).map(class_weights)
    # print("----compute_class_weight:")
    # print(w_train)

    w_train = compute_sample_weight(class_weight='balanced', y=y)
    # print("----compute_sample_weight:")
    # print(w_train)
    
    # Train LightGBM
    lgb_model = LGBMClassifier(**lgb_params, objective='multiclass', num_class=3)
    lgb_model.fit(X, y, sample_weight=w_train)

    # Train XGBoost
    xgb_model = XGBClassifier(**xgb_params, objective='multi:softmax', num_class=3)
    xgb_model.fit(X, y, sample_weight=w_train)

    # Train CatBoost
    cat_model = CatBoostClassifier(**common_params_cat2, objective='MultiClass', classes_count=3)
    cat_model.fit(X, y)

    # Get predictions and ensemble
    lgb_pred = lgb_model.predict_proba(test_X)
    xgb_pred = xgb_model.predict_proba(test_X)
    cat_pred = cat_model.predict_proba(test_X)
    multiclass_pred = np.argmax(lgb_A * lgb_pred + xgb_B * xgb_pred + cat_C * cat_pred, axis=1)
    multiclass_pred_proba = lgb_A * lgb_pred + xgb_B * xgb_pred + cat_C * cat_pred

    multiclass_test_preds = {
        'lgb': lgb_pred,
        'xgb': xgb_pred,
        'cat': cat_pred
    }

    # Feature importance (using LightGBM's importance)
    print("lightgbm feature importance")
    fi_df = pd.DataFrame({'feature': X.columns, 'importance': lgb_model.feature_importances_})
    top10 = fi_df.sort_values(by='importance', ascending=False).head(10)
    feat_str = ", ".join([f"{row['feature']}({int(row['importance'])})" for _, row in top10.iterrows()])
    print(f"[S1] {feat_str}")

    # Feature importance (using XGBoost's importance)
    print("xgboost feature importance")
    fi_df = pd.DataFrame({
        'feature': X.columns,
        'importance': xgb_model.feature_importances_
    })
    top10 = fi_df.sort_values(by='importance', ascending=False).head(10)
    feat_str = ", ".join([f"{row['feature']}({int(row['importance'])})" for _, row in top10.iterrows()])
    print(f"[S1] {feat_str}")

    # Feature importance (using CatBoost's importance)
    print("catboost feature importance")
    fi_df = pd.DataFrame({
        'feature': X.columns,
        'importance': cat_model.get_feature_importance()
    })
    top10 = fi_df.sort_values(by='importance', ascending=False).head(10)
    feat_str = ", ".join([f"{row['feature']}({int(row['importance'])})" for _, row in top10.iterrows()])
    print(f"[S1] {feat_str}")

    # 예측 저장
    submission_final['S1'] = multiclass_pred
    for col in targets_binary:
      submission_final[col] = binary_preds[col]
    submission_final = submission_final[['subject_id', 'sleep_date', 'lifelog_date', 'Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']]
    fname = f"submission_{np.mean(total_avg_f1s)}.csv"
    submission_final.to_csv(fname, index=False)
    print(f"# {fname} 저장 완료")
    print(f"# submission shape:{submission_final.shape}")
    print("================================================")

    # 확률 결과 추가
    submission_proba = submission_final.copy()
    for col in targets_binary:
        for i in range(2):
            submission_proba[f'{col}_class{i}_proba'] = binary_preds_proba[col][:, i]
    for i in range(3):
        submission_proba[f'S1_class{i}_proba'] = multiclass_pred_proba[:, i]
    
    # 저장
    fname_proba = f"submission_with_proba_{np.mean(total_avg_f1s):.4f}.csv"
    submission_proba.to_csv(fname_proba, index=False)
    print(f"# {fname_proba} 저장 완료 (확률 포함)")

    print("\nTop 3 Weight Combinations:")
    for i, (weights, score) in enumerate(zip(top_3_weights, top_3_scores)):
        print(f"Rank {i+1}: lgb_A={weights[0]:.1f}, xgb_B={weights[1]:.1f}, cat_C={weights[2]:.1f} - Score: {score:.4f}")

        # Generate submission with these weights
        lgb_A, xgb_B, cat_C = weights

        # Binary predictions
        for col in targets_binary:
            preds = binary_test_preds[col]
            ensemble_pred = (lgb_A * preds['lgb'] + xgb_B * preds['xgb'] + cat_C * preds['cat'] > 0.5).astype(int)
            submission_final[col] = ensemble_pred
        # Multiclass prediction
        preds = multiclass_test_preds
        ensemble_pred = np.argmax(lgb_A * preds['lgb'] + xgb_B * preds['xgb'] + cat_C * preds['cat'], axis=1)
        submission_final['S1'] = ensemble_pred
        fname = f"submission_top{i+1}_{score:.4f}.csv"
        submission_final.to_csv(fname, index=False)
        print(f"Saved submission to {fname}")

    # Use the best weights for final submission
    best_weights = top_3_weights[0]
    lgb_A, xgb_B, cat_C, = best_weights

    # 모델별 예측결과 비율 비교
    a11 = train_df[['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']].sum()
    a13 = train_df[['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']].apply(len)
    a12 = train_df[['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']].mean()
    a21 = submission_final[['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']].sum()
    a23 = submission_final[['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']].apply(len)
    a22 = submission_final[['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']].mean()
    result = pd.concat([a11, a13, a12, a21, a23, a22], axis=1)
    result.columns = ['학습sum','학습len','학습mean','테스트sum','테스트len','테스트mean']
    print('\n STEP3: 예측결과 비교표')
    display(result)

    # === STEP4: OOF 예측 생성 (train set에 대해) ===

    # n_splits = 10
    mask = train['month'] != 6
    print(f'# k-fold: {n_splits}')
    print(f'# train: {len(y[mask])}')

    oof_f1 = []
    print('\n STEP4: OOF 예측 생성')
    oof_result = train_df[['subject_id', 'sleep_date', 'lifelog_date']].copy()
    for col in targets_binary:
        lgb_params = common_params[col].copy()
        lgb_params['random_state'] = random_state
        
        xgb_params = {
          'n_estimators': 1000,
          "learning_rate": 0.01,
          'reg_lambda': 1,
          'max_depth': 6,
          'n_jobs': -1,
          'subsample': 0.8,
          'colsample_bylevel': 0.8,
          'min_child_weight': 1,
          'max_bin': 200,
          'tree_method': 'hist',
          'random_state': random_state,
        }
        
        y = train_df[col]
        oof_preds = get_oof_predictions(X, y, lgb_params, xgb_params, n_splits=n_splits, is_multiclass=False, early_stop=early_stop)
        oof_result[col] = oof_preds
        f1 = f1_score(y[mask], oof_preds[mask], average='macro')
        oof_f1.append(f1)
        print(f"[OOF - {col}] F1 score: {f1:.4f}")

    # multiclass
    col = 'S1'
    lgb_params = common_params[col].copy()
    lgb_params['random_state'] = random_state
    
    xgb_params = {
      'n_estimators': 1000,
      "learning_rate": 0.01,
      'reg_lambda': 1,
      'max_depth': 6,
      'n_jobs': -1,
      'subsample': 0.8,
      'colsample_bylevel': 0.8,
      'min_child_weight': 1,
      'max_bin': 200,
      'tree_method': 'hist',
      'random_state': random_state,
    }
    
    
    y = train_df[col]
    oof_preds = get_oof_predictions(X, y, lgb_params, xgb_params, n_splits=n_splits, is_multiclass=True, num_class=3, early_stop=early_stop)
    oof_result[col] = oof_preds
    f1 = f1_score(y[mask], oof_preds[mask], average='macro')
    oof_f1.append(f1)
    print(f"[OOF - {col}] F1 score: {f1:.4f}")
    print(f"[OOF] F1 score: {np.mean(oof_f1):.4f}")

    # oof_result 저장
    fname = f"oof_result_{np.mean(total_avg_f1s)}.csv"
    oof_result.to_csv(fname, index=False)
    print(f"# {fname} 저장 완료")

    return submission_final, oof_result

In [154]:
"""
week_type
week_type_lag1
weekday
# X shape: (450, 168)
# test_X shape: (250, 168)

 STEP1: 실험 결과 확인
=============== Validation Results ==============
 평균 F1: 0.6322 / [상세] Q1(기상직후수면질):0.7278 Q2(취침전신체적피로):0.7122 Q3(취침전스트레스):0.6830 S2(수면효율):0.5726 S3(수면잠들기시간):0.6686 S1(S1):0.4293
# 전체 평균 F1: 0.6322
================================================

 STEP2: 전체 데이터로 모델 재학습
====== modoling with 100% train & no valid =====
[Q1] Q1_te2(557), light_night_mean(469), wake_time_ratio(405), wake_time_diff_lag1(340), 통화_time(326), Q1_te(325), sleep_duration_diff(242), activehour_unique_label_count(200), ble_class_others_ratio_worktime(175), wake_time(155)
[Q2] Q2_te2(2791), total_screen_time(351), wake_up_early_minutes(331), speed_le5_max(307), rolling_wake_time_3d(288), rolling_sleep_time_3d(233), sleep_time_diff(224), Q2_te(198), wlight_evening_mean(182), hr_evening_std(176)
[Q3] Q3_te2(2199), light_max(366), sleep_duration_diff_lag1(274), sleep_duration_min(227), sleep_duration_ratio(211), screen_time_vs_avg_pct(187), lat_change(181), all_VEHICLE_minutes(179), sleep_time(177), 통화_time(167)
[S2] S2_te(3725), S2_te2(2773), wake_time_diff_lag1(340), light_max(292), light_night_mean(245), S1_te2(189), ble_class_unknwn_ratio_sleeptime(179), 통화_time(175), sleep_duration_lag1(171), sleep_duration_min(170)
[S3] S3_te(639), light_night_mean(443), S3_te2(336), ble_rssi_mean_afterwork(256), hr_evening_min(242), activehour_unique_label_count(242), ble_class_unknwn_ratio_sleeptime(235), wlight_evening_mean(231), sleep_time_diff_lag1(230), vehicle_minutes(180)
[S1] S1_te2(3990), S1_te(661), sleep_duration_ratio(646), wake_time_ratio(554), sleep_duration_diff(452), sleep_duration_min(429), vehicle_minutes(427), rolling_wake_time_3d(410), speed_le5_max(408), hour_span_minutes(379)
# /content/drive/MyDrive/data/submission_0.6322252622334963.csv 저장 완료
# submission shape:(250, 9)
================================================

 STEP3: 예측결과 비교표
학습sum	학습len	학습mean	테스트sum	테스트len	테스트mean
Q1	223	450	0.4956	131	250	0.5240
Q2	253	450	0.5622	150	250	0.6000
Q3	270	450	0.6000	173	250	0.6920
S1	390	450	0.8667	202	250	0.8080
S2	293	450	0.6511	170	250	0.6800
S3	298	450	0.6622	171	250	0.6840


# k-fold: 5
# train: 392

 STEP4: OOF 예측 생성
[OOF - Q1] F1 score: 0.6976
[OOF - Q2] F1 score: 0.7041
[OOF - Q3] F1 score: 0.6605
[OOF - S2] F1 score: 0.6610
[OOF - S3] F1 score: 0.7106
[OOF - S1] F1 score: 0.5233
[OOF] F1 score: 0.6595
# /content/drive/MyDrive/data/oof_result_0.6322252622334963.csv 저장 완료
"""

# 공통 하이퍼파라미터
common_params = {
  'n_estimators': 5000,
  "learning_rate": 0.01,
  # "shrinkage_rate": 0.12,
  # 'min_data_in_leaf':2,
  # 'bagging_fraction':0.9,
  # 'feature_fraction':0.6,
  'lambda_l1': 5,
  'lambda_l2': 1,
  # 'max_depth': 4,
  'n_jobs': -1,
  'verbosity': -1
}

# 모델별 세부 하이퍼파라미터
best_param_dict = {'Q1': {'learning_rate': 0.1473150575266255,
  'shrinkage_rate': 0.08585454450680065,
  'min_data_in_leaf': 13,
  'bagging_fraction': 0.5900885111562433,
  'feature_fraction': 0.7398526832500182,
  'lambda_l1': 0.7309384079752819,
  'lambda_l2': 0.010419978985191203,
  'max_depth': 2},
 'Q2': {'learning_rate': 0.1433742819325529,
  'shrinkage_rate': 0.4777741359643458,
  'min_data_in_leaf': 11,
  'bagging_fraction': 0.8942012129234453,
  'feature_fraction': 0.3442323511952453,
  'lambda_l1': 0.11108296857244106,
  'lambda_l2': 0.5000682520529595,
  'max_depth': 11},
 'Q3': {'learning_rate': 0.005440413154494791,
  'shrinkage_rate': 0.4869550654391126,
  'min_data_in_leaf': 5,
  'bagging_fraction': 0.992720410336095,
  'feature_fraction': 0.10854085794750301,
  'lambda_l1': 8.765258863766789,
  'lambda_l2': 0.010911793484805324,
  'max_depth': -1},
 'S1': {'learning_rate': 0.19808502263166988,
  'shrinkage_rate': 0.3292477285579064,
  'min_data_in_leaf': 9,
  'bagging_fraction': 0.5929013243246726,
  'feature_fraction': 0.8481981135327139,
  'lambda_l1': 0.010377995886618164,
  'lambda_l2': 0.6226891522266145,
  'max_depth': 10},
 'S2': {'learning_rate': 0.27099064035077214,
  'shrinkage_rate': 0.028901883938906636,
  'min_data_in_leaf': 9,
  'bagging_fraction': 0.8134249396247819,
  'feature_fraction': 0.2321570003912355,
  'lambda_l1': 8.780092357464005,
  'lambda_l2': 9.605716023562762,
  'max_depth': 7},
 'S3': {'learning_rate': 0.14542046442644,
  'shrinkage_rate': 0.3047247759570036,
  'min_data_in_leaf': 10,
  'bagging_fraction': 0.8493532899163512,
  'feature_fraction': 0.7940889257506005,
  'lambda_l1': 9.299803284110112,
  'lambda_l2': 0.12938944891518922,
  'max_depth': 6}
}

# 공통 하이퍼파라미터 대체 (이상한 모델의 경우)
best_param_dict['Q3'] = common_params
best_param_dict['S1'] = common_params
best_param_dict['S2'] = common_params
best_param_dict['S3'] = common_params
best_param_dict['Q1'] = common_params
best_param_dict['Q2'] = common_params

# 전체 평균 F1: 0.6069
# [OOF] F1 score: 0.6491
# [OOF - Q1] F1 score: 0.6913
# [OOF - Q2] F1 score: 0.7078
# [OOF - Q3] F1 score: 0.6432
# [OOF - S2] F1 score: 0.6542
# [OOF - S3] F1 score: 0.7088
# [OOF - S1] F1 score: 0.4895

# 전체 평균 F1: 0.6109
# [OOF] F1 score: 0.6575

# 전체 평균 F1: 0.6172
# [OOF] F1 score: 0.6463

# [수정 전]
# 전체 평균 F1: 0.6308
# [OOF] F1 score: 0.6526

# [수정 후]
# 전체 평균 F1: 0.6322
# [OOF] F1 score: 0.6595


common_params_cat = {
    'iterations': 2000,           # n_estimators에 해당
    'learning_rate': 0.01,
    'l2_leaf_reg': 1,             # reg_lambda에 해당
    'depth': 6,                   # max_depth에 해당
    'thread_count': -1,           # n_jobs에 해당
    # 'subsample': 0.8,
    'rsm': 0.8,                   # colsample_bylevel에 해당
    'min_data_in_leaf': 1,        # min_child_weight에 유사
    'border_count': 200,          # max_bin에 해당
    'task_type': 'CPU',           # 'hist'에 대응
    'random_state' : 41,
    'verbose' : False
}

common_params_cat2 = {
    'iterations': 2000,           # n_estimators에 해당
    'learning_rate': 0.01,
    'class_weights': [1.048, 0.670, 1.807],  # [0, 1, 2] 순서
    'l2_leaf_reg': 1,             # reg_lambda에 해당
    'depth': 6,                   # max_depth에 해당
    'thread_count': -1,           # n_jobs에 해당
    # 'subsample': 0.8,
    'rsm': 0.8,                   # colsample_bylevel에 해당
    'min_data_in_leaf': 1,        # min_child_weight에 유사
    'border_count': 200,          # max_bin에 해당
    'task_type': 'CPU',           # 'hist'에 대응
    'random_state' : 41,
    'verbose' : False
}

submission_final, oof_result = run_basemodel(train, test, valid_ids, best_param_dict, n_splits=5, random_state=41, early_stop=False)

light_week_type_lag1
weekday
week_type
week_type_lag1
activehour_top_bssid
beforebed_top_bssid
# X shape: (450, 247)
# test_X shape: (250, 247)

 STEP1: 실험 결과 확인
=============== Validation Results ==============

Trying weights: lgb_A=0.0, xgb_B=0.0, cat_C=1.0
 Validation Score Q1: 0.6939
 Validation Score Q2: 0.7304
 Validation Score Q3: 0.6438
 Validation Score S2: 0.5474
 Validation Score S3: 0.6686
 Validation Score S1: 0.4429
Average Validation Score: 0.6212

Trying weights: lgb_A=0.0, xgb_B=0.1, cat_C=0.9
 Validation Score Q1: 0.6839
 Validation Score Q2: 0.7405
 Validation Score Q3: 0.6357
 Validation Score S2: 0.5474
 Validation Score S3: 0.6769
 Validation Score S1: 0.4429
Average Validation Score: 0.6212

Trying weights: lgb_A=0.0, xgb_B=0.2, cat_C=0.8
 Validation Score Q1: 0.6839
 Validation Score Q2: 0.7405
 Validation Score Q3: 0.6357
 Validation Score S2: 0.5474
 Validation Score S3: 0.6769
 Validation Score S1: 0.4496
Average Validation Score: 0.6223

Trying weights: lgb

,학습sum,학습len,학습mean,테스트sum,테스트len,테스트mean
Q1,223,450,0.4956,130,250,0.5200
Q2,253,450,0.5622,150,250,0.6000
Q3,270,450,0.6000,177,250,0.7080
S1,390,450,0.8667,204,250,0.8160
S2,293,450,0.6511,158,250,0.6320
S3,298,450,0.6622,169,250,0.6760


# k-fold: 5
# train: 392

 STEP4: OOF 예측 생성
[OOF - Q1] F1 score: 0.7007
[OOF - Q2] F1 score: 0.6937
[OOF - Q3] F1 score: 0.6694
[OOF - S2] F1 score: 0.6608
[OOF - S3] F1 score: 0.6893
[1.05263158 0.67039106 1.79104478 0.67039106 1.05263158 0.67039106
 1.05263158 1.79104478 1.05263158 0.67039106 0.67039106 0.67039106
 0.67039106 0.67039106 0.67039106 0.67039106 0.67039106 0.67039106
 1.05263158 1.05263158 1.05263158 0.67039106 0.67039106 0.67039106
 0.67039106 1.05263158 1.05263158 1.05263158 0.67039106 1.79104478
 1.05263158 0.67039106 1.05263158 0.67039106 1.79104478 1.05263158
 0.67039106 1.79104478 0.67039106 0.67039106 0.67039106 0.67039106
 1.05263158 0.67039106 0.67039106 1.79104478 1.79104478 1.79104478
 1.05263158 0.67039106 1.79104478 0.67039106 0.67039106 1.79104478
 1.79104478 0.67039106 1.79104478 1.79104478 0.67039106 1.05263158
 1.79104478 0.67039106 1.79104478 0.67039106 1.79104478 0.67039106
 1.79104478 0.67039106 0.67039106 0.67039106 0.67039106 0.67039106
 0.67039106 

In [156]:
import pandas as pd
import os
from pathlib import Path

# Reference file
reference_file = '/kaggle/input/dacon-etri-lifelog/submission_0.624161182814381_best.csv'
ref_df = pd.read_csv(reference_file)

# Get all CSV files in data directory
data_dir = Path('/kaggle/working')
csv_files = [f for f in data_dir.glob('submission_*.csv')]

# Store differences for each file
differences = []

for csv_file in csv_files:
    if csv_file.name == os.path.basename(reference_file):
        continue
        
    # Read current file
    current_df = pd.read_csv(csv_file)
    
    # Calculate differences in specified columns
    diff_count = 0
    for col in ['Q1', 'Q2', 'Q3', 'S1', 'S2', 'S3']:
        diff_count += (ref_df[col] != current_df[col]).sum()
    
    differences.append((csv_file.name, diff_count))
    print(f"File: {csv_file.name}, Differences: {diff_count}")

# Sort by difference count and get top 20
differences.sort(key=lambda x: x[1])
print("\nTop 20 files with smallest differences:")
for i, (file_name, diff_count) in enumerate(differences[:20], 1):
    print(f"{i}. {file_name}: {diff_count} differences")

File: submission_top23_0.6211.csv, Differences: 27
File: submission_top54_0.6138.csv, Differences: 36
File: submission_top53_0.6138.csv, Differences: 52
File: submission_top11_0.6223.csv, Differences: 37
File: submission_top33_0.6191.csv, Differences: 24
File: submission_top52_0.6141.csv, Differences: 33
File: submission_top44_0.6166.csv, Differences: 15
File: submission_top19_0.6213.csv, Differences: 22
File: submission_top47_0.6155.csv, Differences: 28
File: submission_top59_0.6112.csv, Differences: 44
File: submission_top4_0.6235.csv, Differences: 14
File: submission_top56_0.6128.csv, Differences: 52
File: submission_top30_0.6197.csv, Differences: 79
File: submission_top43_0.6168.csv, Differences: 23
File: submission_top25_0.6207.csv, Differences: 12
File: submission_top42_0.6170.csv, Differences: 27
File: submission_top12_0.6223.csv, Differences: 93
File: submission_top6_0.6232.csv, Differences: 52
File: submission_top3_0.6237.csv, Differences: 27
File: submission_top35_0.6184.csv,